# Tutorial on CNN architectures for NIR spectral analysis and chemometrics
### We tackle the problem of predicting the concentration of a chemical compound from its NIR spectrum using 1D CNNs. We will use a dataset of NIR spectra and their corresponding concentrations to train and evaluate our models. The tutorial will cover data preprocessing, model building, hyperparameter optimization, and model evaluation.


Dário Passos (dmpassos@ualg.pt)<br>
DeepLight Laboratory, Physics Department <br>
University of Algarve, Faro, Portugal<br>
version 1, 22-07-2026


This optional setup cell lists the pip installations required when running the tutorial in Google Colab. You usually do not need to execute it locally, but it documents the external packages (livelossplot, optuna, GPUtil, tabulate, shap, …) that the rest of the notebook relies on.

In [ ]:
# ## Install some of the necessary packages in google colab
!pip install livelossplot --no-deps
!pip install optuna
!pip install GPUtil
!pip install tabulate
## !pip install umap-learn
!pip install shap
!pip install plotly
!pip install nbformat
!pip install optuna-integration[tfkeras]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Define the path to your data and result folders in Google Drive
# data_path="data"
data_path = '/content/drive/MyDrive/data/cnn'

## 1) Import packages
We import every library used later in the tutorial: scientific Python tooling, plotting utilities, scikit-learn helpers, TensorFlow/Keras layers, and monitoring packages such as livelossplot, Optuna, and GPUtil.



In [ ]:
import os
import sys
from sys import stdout
import logging
import pickle

import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import matplotlib.lines as mlines
import matplotlib.colors as colors
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import cm
import seaborn as sns
import pandas as pd
from IPython.display import clear_output, Image, display

# import scipy.io as sio
from scipy.signal import savgol_filter
from scipy import signal, stats
import tqdm
from tqdm.keras import TqdmCallback
# from itertools import permutations
# import sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
# from sklearn.manifold import TSNE
# from umap import UMAP
from sklearn.model_selection import train_test_split, cross_val_score , KFold
from sklearn.metrics import root_mean_squared_error, mean_squared_error, r2_score
from sklearn.utils import shuffle

import tensorflow as tf
## Set memory growth for GPU to avoid memory allocation issues
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
    
from tensorflow import keras
from tensorflow.keras.activations import elu
from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv1D, Reshape, Dense, Flatten, Lambda
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback,  ModelCheckpoint
from tensorflow.keras.utils import to_categorical, plot_model
# import tensorflow_addons as tfa


## Use liveslossplot for training visualization in real time
from livelossplot import PlotLossesKerasTF
import optuna
import GPUtil
from tabulate import tabulate
import psutil
import platform
import socket
from datetime import datetime

# import umap
# import shap


For future refence, we list the main versions of the main software packages and used hardware
The runtime configuration is inspected here. We select the GPU to use, print timestamps, software versions, CPU/GPU details, and confirm CUDA availability so you can reproduce our environment. 



In [ ]:
## Choose just GPU:1
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Print machine info
print('\n--------  Running @',socket.gethostname(),' using ', platform.platform(),'--------\n' )
# Get current date and time
now = datetime.now()
dt_string = now.strftime("%d/%m/%Y %H:%M:%S")
print("Last run at =", dt_string)

## Print versions and hardware info
print('\n-------- SOFTWARE INFO --------')
print('Python ', sys.version)
print('Tensorflow ', tf.__version__)
# print('Tensorflow add-ons ', tfa.__version__)
print('tqdm ', tqdm.__version__)
print('Numpy ', np.__version__)
print('Pandas', pd.__version__)
print('Optuna ', optuna.__version__)
# print('Scikit-learn ', sklearn.__version__)
# print('livelossplot ', livelossplot.__version__)

## print hardware info
print('\n-------- HARDWARE INFO --------')
# CPU
print('CPU:', platform.processor())
print("\tPhysical cores:", psutil.cpu_count(logical=False))
print("\tTotal cores:", psutil.cpu_count(logical=True))
cpufreq = psutil.cpu_freq()
print(f"\tMax Frequency: {cpufreq.max:.2f}Mhz")
# RAM
print(f'RAM: {int(np.round(psutil.virtual_memory().total / (1024. **3)))} Gb')
# GPU
print('GPU available: ', tf.config.list_physical_devices('GPU'))
print("="*40, "GPU Details", "="*40)
gpus = GPUtil.getGPUs()
list_gpus = []
for gpu in gpus:
    # get the GPU id
    gpu_id = gpu.id
    # name of GPU
    gpu_name = gpu.name
    # get % percentage of GPU usage of that GPU
    gpu_load = f"{gpu.load*100}%"
    # get free memory in MB format
    gpu_free_memory = f"{gpu.memoryFree}MB"
    # get used memory
    gpu_used_memory = f"{gpu.memoryUsed}MB"
    # get total memory
    gpu_total_memory = f"{gpu.memoryTotal}MB"
    # get GPU temperature in Celsius
    gpu_temperature = f"{gpu.temperature} °C"
    gpu_uuid = gpu.uuid
    list_gpus.append((
        gpu_id, gpu_name, gpu_load, gpu_free_memory, gpu_used_memory,
        gpu_total_memory, gpu_temperature, gpu_uuid
    ))

print(tabulate(list_gpus, headers=("id", "name", "load", "free memory", "used memory", "total memory",
                                   "temperature", "uuid")))

print('\nIs CUDA accessible by the GPU? ', tf.test.is_built_with_cuda())

## 2) Help functions
In this section we implement a series of help functions that will be used during the optimization procedure. Run every cell once to ensure that all help functions are loaded.
This helper function seeds Python, NumPy, and TensorFlow generators so that every experiment in the tutorial becomes deterministic across runs.



In [ ]:
## Define random seeds ir order to maintain reproducible results through multiple testing phases
def reproducible_comp():
    os.environ['PYTHONHASHSEED'] = '0'
    np.random.seed(42)
    random.seed(42)
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42)

reproducible_comp()

This next helper function searches for a good learning rate range for the model/dataset combination. It is a good first step to find a bounded interval of learning rates that can be used for hyperparameter optimization latter.

In [ ]:
class LRFinder(tf.keras.callbacks.Callback):
    """
    This callback exponentially adjusts the learning rate after each training batch between `start_lr` and
    `end_lr` for a maximum number of batches: `max_steps`. The loss and learning rate are recorded at each step, allowing
    visually finding a good learning rate as per https://sgugger.github.io/how-do-you-find-a-good-learning-rate.html via
    the `plot` method. The implementation here is a modified version of the original implementation.

    Important:
        Run this on a newly created model and discard that model after the test.
    """

    def __init__(
        self,
        start_lr: float = 1e-7,
        end_lr: float = 1e-1,
        max_steps: int = 300,
        smoothing: float = 0.98,
        diverge_factor: float = 4.0,
        min_steps_before_stopping: int = 20,
    ):
        super().__init__()

        if start_lr <= 0 or end_lr <= start_lr:
            raise ValueError("Require 0 < start_lr < end_lr.")

        max_steps = int(max_steps)
        min_steps_before_stopping = int(min_steps_before_stopping)

        if max_steps < 2:
            raise ValueError("max_steps must be at least 2.")
        if not 0.0 <= smoothing < 1.0:
            raise ValueError("smoothing must satisfy 0 <= smoothing < 1.")
        if diverge_factor <= 1.0:
            raise ValueError("diverge_factor must be greater than 1.")
        if min_steps_before_stopping < 0:
            raise ValueError("min_steps_before_stopping cannot be negative.")

        self.start_lr = float(start_lr)
        self.end_lr = float(end_lr)
        self.max_steps = max_steps
        self.smoothing = float(smoothing)
        self.diverge_factor = float(diverge_factor)
        self.min_steps_before_stopping = min_steps_before_stopping

        self.step = 0
        self.avg_loss = 0.0
        self.best_loss = np.inf
        self.current_lr = None

        self.lrs = []
        self.losses = []

    def on_train_begin(self, logs=None):
        self.step = 0
        self.avg_loss = 0.0
        self.best_loss = np.inf
        self.current_lr = None

        self.lrs = []
        self.losses = []

    def _learning_rate_at_step(self, step: int) -> float:
        progress = step / max(self.max_steps - 1, 1)

        return self.start_lr * (
            self.end_lr / self.start_lr
        ) ** progress

    def _set_learning_rate(self, lr: float):
        learning_rate = self.model.optimizer.learning_rate

        if hasattr(learning_rate, "assign"):
            learning_rate.assign(lr)
        else:
            self.model.optimizer.learning_rate = lr

    def on_train_batch_begin(self, batch, logs=None):
        self.current_lr = self._learning_rate_at_step(self.step)
        self._set_learning_rate(self.current_lr)

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        loss = logs.get("loss")
        if loss is None:
            raise RuntimeError("LRFinder requires a compiled model with a loss.")
        loss = float(loss)
        if not np.isfinite(loss):
            self.model.stop_training = True
            self.model.reset_metrics()
            return
        self.avg_loss = (
            self.smoothing * self.avg_loss
            + (1.0 - self.smoothing) * loss
        )
        bias_correction = 1.0 - self.smoothing ** (self.step + 1)
        smooth_loss = self.avg_loss / bias_correction
        lr = self.current_lr

        self.lrs.append(lr)
        self.losses.append(smooth_loss)

        if smooth_loss < self.best_loss:
            self.best_loss = smooth_loss

        if (
            self.step >= self.min_steps_before_stopping
            and smooth_loss > self.diverge_factor * self.best_loss
        ):
            self.model.stop_training = True

        self.step += 1

        if self.step >= self.max_steps:
            self.model.stop_training = True

        # Important for an LR-finder-only run:
        # prevents Keras loss metrics accumulating across batches.
        self.model.reset_metrics()

    def suggestions(self, skip_start: int = 10, skip_end: int = 5):
        """
        Returns two heuristic LR suggestions:

        steepest_descent:
            LR corresponding to the steepest decrease in loss.

        one_tenth_minimum:
            One tenth of the LR at the minimum smoothed loss.
        """
        lrs = np.asarray(self.lrs, dtype=float)
        losses = np.asarray(self.losses, dtype=float)

        if len(lrs) <= skip_start + skip_end + 2:
            raise RuntimeError("Not enough LR-finder observations.")

        stop = len(lrs) - skip_end

        selected_lrs = lrs[skip_start:stop]
        selected_losses = losses[skip_start:stop]

        # A short centred moving average suppresses the batch/epoch ripple that
        # can otherwise make the steepest-gradient heuristic select the left edge.
        window = min(9, len(selected_losses) - 2)
        if window % 2 == 0:
            window -= 1

        if window >= 3:
            kernel = np.ones(window, dtype=float) / window
            stable_losses = np.convolve(selected_losses, kernel, mode="valid")
            offset = window // 2
            stable_lrs = selected_lrs[offset:-offset]
        else:
            stable_losses = selected_losses
            stable_lrs = selected_lrs

        gradient = np.gradient(stable_losses, np.log10(stable_lrs))

        steepest_index = np.argmin(gradient)
        minimum_index = np.argmin(stable_losses)

        return {
            "steepest_descent": float(stable_lrs[steepest_index]),
            "one_tenth_minimum": float(stable_lrs[minimum_index] / 10.0),
            "minimum_loss_lr": float(stable_lrs[minimum_index]),
        }

    def plot(self, skip_start: int = 10, skip_end: int = 5, steepest_lr=None):
        lrs = np.asarray(self.lrs)
        losses = np.asarray(self.losses)

        stop = len(lrs) - skip_end if skip_end > 0 else len(lrs)

        fig, ax = plt.subplots(figsize=(8, 6))

        ax.plot(lrs[skip_start:stop],  losses[skip_start:stop],  linewidth=2)
        if steepest_lr is None:
            steepest_lr = self.suggestions(
                skip_start=skip_start,
                skip_end=skip_end,
            )["steepest_descent"]
        steepest_lr = float(steepest_lr)
        ax.axvline(
            steepest_lr,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label=f"Steepest descent = {steepest_lr:.2e}",
        )
        ax.set_xscale("log")
        ax.set_xlabel("Learning rate")
        ax.set_ylabel("Smoothed training loss")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()

        return fig, ax

We predefine a couple of auxiliary functions to compute error metrics and make prediction plots.
All the utility functions used throughout the notebook are defined here: metrics/plotting helpers, PLS optimisation routines, and wrappers that we will reuse whenever we evaluate models.

In [ ]:
## Function to compute metrics and make prediction plots using train and test data
def plot_prediction2(Y_train, Y_test, Y_train_pred, Y_test_pred, title, savefig=False, figname=None):
    """Compute regression metrics for train/test predictions and generate a diagnostic plot.
    @author: Dario Passos
    Parameters
    ----------
    Y_train : array-like
        Observed target values for the training data.
    Y_test : array-like
        Observed target values for the test data.
    Y_train_pred : array-like
        Model predictions corresponding to `Y_train`.
    Y_test_pred : array-like
        Model predictions corresponding to `Y_test`.
    title : str
        Title applied to the plot.
    savefig : bool, optional
        Save the figure instead of displaying it when True.
    figname : str, optional
        Path used when saving the figure.
    
    Returns
    -------
    None
    """

    ## Compute train error scores
    score_p0 = r2_score(Y_train, Y_train_pred)
    mse_p0 = mean_squared_error(Y_train, Y_train_pred)
    rmse_p0 = np.sqrt(mse_p0)

    ## Compute test error scores
    score_p2 = r2_score(Y_test, Y_test_pred)
    mse_p2 = mean_squared_error(Y_test, Y_test_pred)
    rmse_p2 = np.sqrt(mse_p2)

    print('ERROR METRICS: \t TRAIN  \t\t TEST')
    print('------------------------------------------------------')
    print('R2:   \t\t %5.3f  \t\t %5.3f'  % (score_p0, score_p2 ))
    print('RMSE: \t\t %5.3f  \t\t %5.3f' % (rmse_p0, rmse_p2))

    #### Plot regression for model predicted data
    ## Get plot limits
    Y = np.concatenate([Y_train, Y_test])

    rangey = np.max(Y) - np.min(Y)
    rangex = np.max(Y) - np.min(Y)
    ## x=y line and +- 1std upper and lower bowndaries
    xy_x=np.ravel([np.min(Y)-0.1*rangex, np.max(Y)+0.1*rangex])
    xy_y=np.ravel([np.min(Y)-0.1*rangey, np.max(Y)+0.1*rangey])

    plt.figure(figsize=(5,5))
    z = np.polyfit(np.ravel(Y_test), np.ravel(Y_test_pred), 1)
    print('Fit result: Y=',z[1], ' + ', z[0],' * X')
    ax = plt.subplot(aspect=1)
    ax.plot(xy_x, xy_y, 'k--', linewidth=2, label=None)
    ax.scatter(Y_train, Y_train_pred, c='gray', marker='o', s=20, alpha=0.66, label='train')
    ax.scatter(Y_test,Y_test_pred, s=40, marker='o', facecolors='None', edgecolors='r', label='test')
    # Calculate the range of x-axis based on Y_train and Y_test
    x_min = min(np.min(Y_train), np.min(Y_test))
    x_max = max(np.max(Y_train), np.max(Y_test))
    # Create an array spanning the range of x-axis
    x_range = np.linspace(x_min, x_max, num=100)
    ax.plot(x_range, z[1]+z[0]*x_range, c='blue', linewidth=2,label='linear fit')
    plt.xlim(xy_x)
    plt.ylim(xy_y)
    # ax.plot(x_range, x_range, 'k--', linewidth=1.5, label='y=x')
    plt.ylabel('Predicted')
    plt.xlabel('Measured')
    plt.title(title)
    plt.legend(loc=4, frameon=False)

    # Print the scores on the plot
    plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.1*rangey, 'R$^{2}=$ %5.2f'  % score_p2, fontsize=13)
    plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.15*rangey, 'RMSE: %5.2f' % rmse_p2, fontsize=13)
    if savefig==True:
        plt.savefig(figname, dpi=150)
        print('Figure saved')
    else:
        plt.show()
    return


## Function to compute metrics and make prediction plots (custom version of previous function)
def plot_Y_prediction(Y, Y_pred, title, ax, savefig=False, figname=None):
    """Plot measured versus predicted targets and annotate simple regression metrics.
    @author: Dario Passos
    Parameters
    ----------
    Y : array-like
        Observed target values.
    Y_pred : array-like
        Predicted target values from the model.
    title : str
        Title applied to the plot.
    ax : matplotlib.axes.Axes
        Axes that receives the scatter plot and fitted line.
    savefig : bool, optional
        Save the figure instead of displaying it when True.
    figname : str, optional
        Path used when saving the figure.

    Returns
    -------
    None
    """
    ## Compute train error scores
    score_p0 = r2_score(Y, Y_pred)
    mse_p0 = mean_squared_error(Y, Y_pred)
    rmse_p0 = np.sqrt(mse_p0)
#     print('R2: \t\t %5.3f '  % (score_p0))
#     print('RMSE: \t\t %5.3f' % (rmse_p0))
    ## Plot regression for PLS predicted data
    rangey = max(Y) - min(Y)
    rangex = max(Y_pred) - min(Y_pred)
    fig=plt.figure(figsize=(3,3))
    z = np.polyfit(np.ravel(Y), np.ravel(Y_pred), 1)
    print('Fit result: Y=',z[1], ' + ', z[0],' * X')
    # ax = plt.subplot(aspect=1)
    ax.scatter(Y,Y_pred,c='k',marker='o',s=10, alpha=0.6)
    ax.plot(Y, z[1]+z[0]*Y, c='blue', linewidth=2,label='linear fit')
    ax.plot(Y, Y, 'k--', linewidth=1.5, label='y=x')
    ax.set_ylabel('Predicted')
    ax.set_xlabel('Measured')
    ax.set_title(title)
#     plt.legend(loc=4)
    # Print the scores on the plot
    ax.text(min(Y_pred)+0.02*rangex, max(Y)-0.1*rangey, 'R$^{2}=$ %5.3f'  % score_p0)
    ax.text(min(Y_pred)+0.02*rangex, max(Y)-0.15*rangey, 'RMSE: %5.3f' % rmse_p0)
    if savefig==True:
        plt.savefig(figname, dpi=150)
        print('Figure saved')
    else:
        plt.show()
    return




### Functions for computing and optimizing PLS models ############

# from chemometrics_analysis_help import *

def error_metrics(y_true0, y_pred0):
    """Compute standard regression metrics for a prediction/ground-truth pair.
    @author: Dario Passos
    Parameters
    ----------
    y_true0 : array-like
        Ground-truth target values.
    y_pred0 : array-like
        Predicted target values.

    Returns
    -------
    Tuple[float, float, float, float, float]
        R-squared, RMSE, prediction gain, coefficient of variation, and SDR.
    """
    y_true=np.ravel(y_true0)
    y_pred=np.ravel(y_pred0)
    ## R squared R2 (based on the Pearson correlation)
    R2 = stats.pearsonr(y_true.squeeze(),y_pred.squeeze())[0]**2
    ## Root Mean Squared Error (RMSE)
    RMSE = np.sqrt(mean_squared_error(y_true, y_pred))
    ## Prediction Gain (PG)
     # initialize PG vector with the mean value
    PG0 = np.zeros(len(y_pred)) + np.mean(y_true)
     # Now we compute the rms error between this preciction (mean value) and the validation set
    PG0_MSE= np.sqrt(mean_squared_error(y_true, PG0))
    PG= PG0_MSE / RMSE
    ## Coefficient of Variation (CVAR)
    CVAR = np.round(100.*RMSE/np.mean(y_true),2)
    SDR = np.std(y_true) / RMSE
    return np.round(R2,3), np.round(RMSE,3), np.round(PG,3), np.round(CVAR,3), np.round(SDR,3)


def pls_optimization_cv_stop2(x_train, y_train, nmax=20, Nfolds=5, plot_opt=False, stop_criteria=0.01):
    """Computes the optimal number of LVs for a PLS model using cross-validation using as stop criteria
        the LV that produces a gain in the CV RMSE lower than 1% (default) of the previous LV.
    @author: Dario Passos
    Parameters
    ----------
    x_train : array-like
        Training predictors used to fit each candidate model.
    y_train : array-like
        Training targets aligned with `x_train`.
    nmax : int, optional
        Maximum number of components to evaluate.
    Nfolds : int, optional
        Number of folds used during cross-validation.
    plot_opt : bool, optional
        Draw the CV RMSE trace when True.
    stop_criteria : float, optional
        Relative RMSE improvement threshold that triggers early stopping.

    Returns
    -------
    Tuple[int, float]
        Selected LV count and the corresponding CV RMSE.

    """
    ## List to store the CV RMSE for each LV
    cv_rmse=[]

    print('\nComputing optimal number of LVs for PLS model in the range 1 to {}...\n'.format(nmax))
    component = np.arange(1, nmax+1)
    previous_cv_rmse = None
    bestLV_stop = None

    print('Stop criteria: {}% gain in RMSE'.format(stop_criteria*100))

    for i in component:
        pls = PLSRegression(n_components=i, scale=True)
        cv_score=cross_val_score(pls, x_train, y_train, cv=KFold(Nfolds, shuffle = True, random_state=42),\
                         scoring='neg_mean_squared_error', error_score=0)
        current_cv_rmse = np.round(np.sqrt(-np.mean(cv_score)),3)
        ## Check if the current CV RMSE is less than 1% of the previous CV RMSE
        if (previous_cv_rmse is not None) and (current_cv_rmse <= np.min(cv_rmse)):
            percent_diff = abs((current_cv_rmse - previous_cv_rmse) / previous_cv_rmse)
            # print(f'Compute difference percentage between current LV={i} and previous LV={i-1} CV RMSE -> {np.round(percent_diff*100,3)}%')
            if percent_diff <= stop_criteria and bestLV_stop is None:
                print(f"Stopping criteria reached, {np.round(percent_diff*100,3)}%. Saving component number.")
                ## The previous LV is the last one that adds more than 1% of gain in RMSE
                bestLV_stop = i-1
                ## Save the RMSE of the previous LV
                RMSE_stop = previous_cv_rmse
        previous_cv_rmse = current_cv_rmse
        cv_rmse.append(current_cv_rmse)
        criterion_flag = '1% gain'

    ## if the 1% criteria returns no LV, then the bestLV_stop is the one where the RMSE is minimum
    if bestLV_stop is None:
        print(f'Stop criteria of {stop_criteria*100}% gain in RMSE not reached. Using minimum RMSE.')
        bestLV_stop = np.argmin(cv_rmse)+1
        RMSE_stop = np.min(cv_rmse)
        criterion_flag = 'minimum RMSE'

    RMSE_best = np.round(np.min(RMSE_stop),3)
    print(f'Suggested number of LV based on {Nfolds}-fold CV RMSE using {criterion_flag}: {bestLV_stop}')
    print(f'{Nfolds} CV RMSE: {RMSE_best}')
    stdout.write("\n")
    if plot_opt is True:
        plt.figure(figsize=(9,3))
        ax1=plt.subplot()
        ax1.plot(component[:len(cv_rmse)], np.array(cv_rmse), '-v', color = 'blue', mfc='blue')
        if bestLV_stop is not None:
            ax1.plot(component[bestLV_stop-1], [RMSE_best], 'P', ms=10, mfc='red',label='LV chosen')
        plt.xlabel('Number of PLS components')
        plt.ylabel('Mean of '+str(Nfolds)+'CV RMSE ')
        ax1.axvline(x=bestLV_stop, color='red', lw=1,linestyle='--')
        ax1.set_xticks(component)
        plt.xlim(0, nmax+1)
        # plt.title('# PLS components')
        plt.legend()
        plt.grid(alpha=0.33)
        plt.show()
    return bestLV_stop, RMSE_best




def pls_prediction_metrics2(l, x_train, y_train, x_test, y_test, xname, yname, lv, verbose=True, plot_pred=False, plot_vip=False):
    """Fit a PLS model with a fixed LV count and report regression diagnostics and plots.
    @author: Dario Passos
    Parameters
    ----------
    l : array-like
        Wavelength axis used when plotting VIP scores.
    x_train : array-like
        Predictors for the training set.
    y_train : array-like
        Targets for the training set.
    x_test : array-like
        Predictors for the test set.
    y_test : array-like
        Targets for the test set.
    xname : str
        Label used for the predictor axis in plots.
    yname : str
        Label used for the target axis in plots.
    lv : int
        Number of latent variables to retain in the PLS model.
    verbose : bool, optional
        Print tabulated metric output when True.
    plot_pred : bool, optional
        Draw measured-versus-predicted scatter plots.
    plot_vip : bool, optional
        Draw VIP scores for each wavelength.

    Returns
    -------
    Tuple
        Train/test metrics and predictions in the order returned by `error_metrics`.
    """
    
    ## Define PLS with suggested optimal number of components and fit train data
    pls1 = PLSRegression(n_components=lv, scale=True)

    ## Fit PLS model to train data
    pls1.fit(x_train, y_train)

    ## Get predictions for train and test sets
    y_train_pred = pls1.predict(x_train)
    y_test_pred = pls1.predict(x_test)

    ## Compute error metrics
    R2_train, RMSE_train, PG_train, CVAR_train, SDR_train = error_metrics(y_train, y_train_pred)
    R2_test, RMSE_test, PG_test, CVAR_test, SDR_test = error_metrics(y_test, y_test_pred)

    if verbose == True:
        print('\nError metrics for best PLS model with LV = {}'.format(lv))
        print('METRIC \t TRAIN \t TEST ')
        print('R2     \t {:0.3f}\t {:0.3f}'.format(R2_train,R2_test))
        print('RMSE   \t {:0.3f}\t {:0.3f}'.format(RMSE_train,RMSE_test))
        # print('PG   \t {:0.3f}\t {:0.3f}'.format(PG_train,PG_test))
        print('CVAR   \t {:0.3f}\t {:0.3f}'.format(CVAR_train,CVAR_test))
        print('SDR  \t {:0.3f}\t {:0.3f}'.format(SDR_train,SDR_test))

    ## Plots: MSE vs. PLS LV and regression for best PLS model
    # Get plot limits
    Y = np.concatenate([y_train, y_test])

    rangey = np.max(Y) - np.min(Y)
    rangex = np.max(Y) - np.min(Y)

    # x=y line and +- 1std upper and lower bowndaries
    xy_x=np.ravel([np.min(Y)-0.1*rangex, np.max(Y)+0.1*rangex])
    xy_y=np.ravel([np.min(Y)-0.1*rangey, np.max(Y)+0.1*rangey])

    xy_y_up=xy_y+np.std(Y)
    xy_y_down=xy_y-np.std(Y)

    if plot_pred is True:
        ## linear fit to predicted test data
        plt.figure(figsize=(5,5))
        # plt.title(yname+' prediction using '+xname+' data')

        ## fit the test data
        z = np.polyfit(np.ravel(y_test), np.ravel(y_test_pred), 1)
        print('Fit result: Y=',z[1], ' + ', z[0],' * X')
        ax = plt.subplot()
        ax.plot(xy_x, xy_y, 'k--', linewidth=2, label = None)
        # plt.fill_between(xy_x, xy_y_down, xy_y_up, alpha=0.2)
        ax.scatter(y_train,y_train_pred,c='gray',s=26, marker='o', alpha=0.66, label='Train')
        ax.scatter(y_test,y_test_pred,s=40, marker='o', facecolors='None', edgecolors='r', label='Test')
        x_range = np.linspace(np.min(Y), np.max(Y), num=100)
        ax.plot(x_range, z[1]+z[0]*x_range, c='blue', linewidth=3, label = None)
        plt.xlim(xy_x)
        plt.ylim(xy_y)
        plt.ylabel('Predicted '+yname, fontsize=10)
        plt.xlabel('Measured '+yname, fontsize=10)
        plt.legend(loc=4)
        # Print the test error metrics on the plot
        plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.1*rangey, 'R$^{2}=$ %5.2f'  % R2_test, fontsize=13)
        plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.15*rangey, 'RMSE: %5.2f' % RMSE_test, fontsize=13)
        # plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.2*rangey, 'PG: %5.2f' % PG_test, fontsize=13)
        # plt.text(np.min(xy_x)+0.05*rangex, np.max(xy_y)-0.25*rangey, 'CVar: %5.2f%%' % CVAR_test, fontsize=13)
        plt.show()

    if plot_vip is True:
        pls_vip=vip(pls1)
        fig, ax = plt.subplots(figsize=(12,3))
        # plt.title('PLS VIP scores ')
        plt.ylabel('VIP score', fontsize=14)
        plt.xlabel('Wavelength (nm)', fontsize=14)
        ax.plot(l,pls_vip,'k',label='VIP scores')
        ax.set_ylim(np.min(pls_vip), np.max(pls_vip))
        plt.axhline(1,color='k', linestyle='--',linewidth=0.75)
        plt.legend()
        plt.show()

    return R2_train, RMSE_train, PG_train, CVAR_train, SDR_train, R2_test, RMSE_test, PG_test, CVAR_test, SDR_test, y_train_pred, y_test_pred



def pls_explained_variance(pls, X, Y_true, do_plot=True):
    """Compute per-component and overall R-squared for a fitted PLS model.
    @author: Dario Passos
    Parameters
    ----------
    pls : sklearn.cross_decomposition.PLSRegression
        Trained PLS model to analyse.
    X : array-like
        Predictor matrix used to compute projections.
    Y_true : array-like
        Ground-truth targets corresponding to `X`.
    do_plot : bool, optional
        Draw a bar plot of component-wise R-squared when True.

    Returns
    -------
    Tuple[np.ndarray, float]
        Component-wise R-squared values and the overall R-squared.
    """

    r2 = np.zeros(pls.n_components)
    x_transformed = pls.transform(X) # Project X into low dimensional basis
    for i in range(0, pls.n_components):
        Y_pred = (np.dot(x_transformed[:, i][:, np.newaxis],
                         pls.y_loadings_[:, i][:, np.newaxis].T) * pls._y_std
                  + pls._y_mean)
        r2[i] = r2_score(Y_true, Y_pred)
        overall_r2 = r2_score(Y_true, pls.predict(X))  # Use all components together.

    if do_plot:
        plt.figure(figsize=(5,5))
        component = np.arange(pls.n_components) + 1
        plt.bar(component, r2)
        plt.xticks(component)
        plt.xlabel('Number of PLS components')
        plt.ylabel('Explained variance')
        # plt.title(f'Summed individual r2: {np.sum(r2):.3f}, '
        #           f'Overall r2: {overall_r2:.3f}')
        plt.show()

    return r2, overall_r2


def vip(model):
    """Compute variable importance in projection (VIP) scores for a fitted PLS model.
    @author: Dario Passos
    Parameters
    ----------
    model : sklearn.cross_decomposition.PLSRegression
        PLS model providing X scores, weights, and Y loadings.
    Returns
    -------
    np.ndarray
        VIP score for each predictor variable.
    """
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_

    n_features, n_components = w.shape

    # Amount of Y variance associated with each PLS component.
    component_ss = np.sum(t ** 2, axis=0) * np.sum(q ** 2, axis=0)
    total_ss = np.sum(component_ss)

    if total_ss == 0:
        raise ValueError("Cannot compute VIP scores: total explained Y variance is zero.")

    # Normalize each PLS weight vector, then calculate all VIP scores at once.
    normalized_weights = w / np.linalg.norm(w, axis=0, keepdims=True)
    return np.sqrt(
        n_features * ((normalized_weights ** 2) @ component_ss) / total_ss
    )



def snv(x_data):
    """ 
    Computes the Standard Normal Variate (SNV) from the full range of the spectrum
    -----------------------------------------
    x_data: numpy array (n x m) with n rows (samples) and m columns (features)
    -----------------------------------------
    """
    # Define a new array and populate it with the corrected data  
    data_snv = np.zeros_like(x_data)
    for i in range(x_data.shape[0]):
         # Apply correction
        data_snv[i,:] = (x_data[i,:] - np.mean(x_data[i,:])) / np.std(x_data[i,:])
    return data_snv


# Multiplicative Scattering Correction using mean spectra as reference
def msc(input_data, reference=None):
    ''' 
    Perform Multiplicative Scattering Correction (MSC).
    If no reference is introduced, the mean spectra is used as reference.
    The function can be modified to return the mean spectra for use on other data (see last line)
    -----------------------------------------
    input_data: numpy array (n x m), with n rows (samples) and m columns (features)
    reference: numpy array (1 x m)
    -----------------------------------------
    '''
    # mean centre correction
    input_data_centered = np.zeros_like(input_data)
    for i in range(input_data.shape[0]):
        input_data_centered[i,:] = input_data[i,:] - input_data[i,:].mean()
    # Get the reference spectrum. If not given, estimate it from the mean    
    if reference is None:    
        # Calculate mean
        ref = np.mean(input_data_centered, axis=0)
    else:
        ref = reference
    # Define a new array and populate it with the corrected data    
    data_msc = np.zeros_like(input_data_centered)
    for i in range(input_data_centered.shape[0]):
        # Run regression
        fit = np.polyfit(ref, input_data_centered[i,:], 1, full=True)
        # Apply correction
        data_msc[i,:] = (input_data_centered[i,:] - fit[0][1]) / fit[0][0] 
    return data_msc
#     return data_msc, ref ## return the reference spectrum as well

Set parameters for graphics formating
Plotting defaults (fonts, colours, grid styles, etc.) are configured so that every subsequent figure in the tutorial shares the same visual style.



In [ ]:
## Graphics settings
## Setting the font sizes for comming figures
plt.style.use("default")
SMALL_SIZE = 10
MEDIUM_SIZE = 12
BIGGER_SIZE = 14

## uncomment for Latex graphics formating
# plt.rcParams.update({
#     "text.usetex": True,
#     "font.family": "serif",
#     "font.sans-serif": ["Times"]})

# plt.rc('text', usetex=True)
plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

## 3) Data loading

### 3.1) Loading the datasets
All datasets were pre-formated with the first n-1 columns being the NIR spectra and the last column being the target variable. The first column is sample ID.

In this tutorial we make use of 3 different datasets:

**Tecator moisture** (Karin Thente, Tecator AB) <br>
NIR spectra, 850 - 1050 nm, of meats and moisture reference values. Downloaded from: \url{https://lib.stat.cmu.edu/datasets/tecator}

**tomato4 ssc**, (Ibañez, G. *et al* 2019) <br>
NIR spectra, 902 - 2094 nm, from a tomato variety and SSC references. The original dataset was split according to the 5 different types of tomato products. Downloaded from: \url{https://zenodo.org/records/10633732}. DOI: https://doi.org/10.5281/zenodo.10633732

**CEOT pears 2019 brix** (Passos, D. *et al* 2019) <br> 
NIR spectra, 500 - 1100 nm from intact pears (var. 'Rocha') and reference SSC values (CEOT-UAlg). Available upon request to the author. DOI: https://doi.org/10.3390/s19235165


In [ ]:
## Import the train and test datasets into a Pandas Dataframe object. Double check the path to the files

# Tecator
tecator_train = pd.read_csv(os.path.join(data_path, 'tecator/data_train.csv'), index_col=0)
tecator_test = pd.read_csv(os.path.join(data_path, 'tecator/data_test.csv'), index_col=0)
# Tomato
tomato_train = pd.read_csv(os.path.join(data_path, 'tomato/data_train.csv'), index_col=0)
tomato_test = pd.read_csv(os.path.join(data_path, 'tomato/data_test.csv'), index_col=0)
# Pear
pear_train = pd.read_csv(os.path.join(data_path, 'pears/data_train.csv'), index_col=0)
pear_test = pd.read_csv(os.path.join(data_path, 'pears/data_test.csv'), index_col=0)


In [ ]:
# Taka a quick look at the data (tecator)
tecator_test.head()  # head() shows the first 5 rows of the dataframe. You can also use .tail() to see the last 5 rows.

In [ ]:
# Check some basic information about the dataset, such as the number of rows and columns, data types, and missing values.
tecator_train.info()

# Check statistical summary of the dataset, including count, mean, std, min, 25%, 50%, 75%, and max for each column.
tecator_train.describe()

Before we proceed with the modelling steps, we separate X (spectra) and y (target variable) for each dataset. The first column is sample ID, the last column is the target variable, and the remaining columns are the NIR spectra.

In [ ]:
### Define the spectra... +
tecator_x_train = tecator_train.iloc[:, :-1] # iloc[:, :-1] selects all rows and all columns except the last one
tecator_x_test = tecator_test.iloc[:, :-1]
tomato_x_train = tomato_train.iloc[:, :-1]
tomato_x_test = tomato_test.iloc[:, :-1]
pear_x_train = pear_train.iloc[:, :-1]
pear_x_test = pear_test.iloc[:, :-1]

### ... and the y values 
# we explicitly define the y values for each dataset, which are the target variables 
# we want to predict. We could have used the iloc method to select the last 
# column data.iloc[:,-1]
tecator_y_train = tecator_train['Moisture'] 
tecator_y_test = tecator_test['Moisture']
tomato_y_train = tomato_train['SSC']
tomato_y_test = tomato_test['SSC']
pear_y_train = pear_train['brix']
pear_y_test = pear_test['brix']


## Extract the wavelength values from the column names of the training data for plotting purposes
# convert w list to float array rounded to 2 decimal places
tecator_w = np.array([round(float(i), 2) for i in tecator_train.columns.values[:-1]])
tomato_w = np.array([round(float(i), 2) for i in tomato_train.columns.values[:-1]])
pear_w = np.array([round(float(i), 2) for i in pear_train.columns.values[:-1]])

## Print datasets main information: train and tes size, number of features, and wavelength range
print('Tecator dataset: train size = {}, test size = {}, number of features = {}, wavelength range = {}-{} nm'.format(tecator_x_train.shape[0], tecator_x_test.shape[0], tecator_x_train.shape[1], np.min(tecator_w), np.max(tecator_w)))
print('Tomato dataset: train size = {}, test size = {}, number of features = {}, wavelength range = {}-{} nm'.format(tomato_x_train.shape[0], tomato_x_test.shape[0], tomato_x_train.shape[1], np.min(tomato_w), np.max(tomato_w)))
print('Pear dataset: train size = {}, test size = {}, number of features = {}, wavelength range = {}-{} nm'.format(pear_x_train.shape[0], pear_x_test.shape[0], pear_x_train.shape[1], np.min(pear_w), np.max(pear_w)))

Take a look at the spectra

In [ ]:
## create a grid with 3 plots side by side to visualize the spectra of the three datasets
plt.figure(figsize=(16,4))
plt.subplot(1,3,1) # 1 row, 3 columns, 1st subplot
plt.plot(tecator_w, tecator_x_test.T, alpha=0.5)
plt.title('Tecator spectra')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance')
plt.subplot(1,3,2) # 1 row, 3 columns, 2nd subplot
plt.plot(tomato_w, tomato_x_test.T, alpha=0.5)
plt.title('Tomato spectra')
plt.xlabel('Wavelength (nm)')   
plt.ylabel('Absorbance')
plt.subplot(1,3,3) # 1 row, 3 columns, 3rd subplot
plt.plot(pear_w, pear_x_test.T, alpha=0.5)
plt.title('Pear spectra')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Absorbance')
plt.show()

### 3.2) Advanced Visualization examples

Color the spectra according to the target variable. This is a useful visualization technique to inspect the relationship between the spectral data and the target variable. We will use a colormap to represent the target variable values, allowing us to see how the spectra vary with respect to the target.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(10,6))

# Define colormap: Green to Red
# 'RdYlGn_r' Red at high values and Green at low values
cmap = plt.get_cmap('RdYlGn_r') 
norm = mcolors.Normalize(vmin=pear_y_test.min(), vmax=pear_y_test.max())

# Iterate through pear_x_test and plot each spectrum with the corresponding color
for i in range(len(pear_x_test)):
    brix_val = pear_y_test.iloc[i]
    ax.plot(pear_w, pear_x_test.iloc[i], color=cmap(norm(brix_val)), alpha=0.60)
# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Brix')
ax.set_xlabel('Wavelength')
ax.set_title('Pear spectra Colored by Brix')
plt.show()

Inspecting the y variables

In [ ]:
plt.figure(figsize=(16,3))
plt.subplot(1, 3, 1)
plt.plot(np.arange(len(tecator_y_train)), tecator_y_train, 'ro', markersize=3, label='Train')
plt.plot(np.arange(len(tecator_y_train), len(tecator_y_train)+len(tecator_y_test)), tecator_y_test, 'bo', markersize=3,label='Test')
plt.title('Tecator Moisture values')
plt.ylabel('Moisture (%)')
plt.xlabel('Sample number')
plt.legend()
plt.subplot(1, 3, 2)
plt.plot(np.arange(len(tomato_y_train)), tomato_y_train, 'ro', markersize=3, label='Train')
plt.title('Tomato SSC values')
plt.ylabel('SSC')
plt.xlabel('Sample number')
plt.plot(np.arange(len(tomato_y_train), len(tomato_y_train)+len(tomato_y_test)), tomato_y_test, 'bo', markersize=3, label='Test')
plt.subplot(1, 3, 3)
plt.plot(np.arange(len(pear_y_train)), pear_y_train, 'ro', markersize=3, label='Train')
plt.title('Pear Brix values')
plt.ylabel('Brix')
plt.xlabel('Sample number')
plt.plot(np.arange(len(pear_y_train), len(pear_y_train)+len(pear_y_test)), pear_y_test, 'bo', markersize=3, label='Test')
plt.legend()
plt.show()

### 3.3) PCA decomposition and Hotelling T^2/Q-residuals diagnostics (outlier detection)

In this brief section we perform a PCA decomposition of the training data and compute Hotelling T^2 and Q-residuals diagnostics for the PCA model. This is useful to identify potential outliers in the dataset before proceeding with the modelling steps.

In [ ]:
## import PCA library from sklearn
from sklearn.decomposition import PCA

# Define the dataset we want to perform PCA on.
x_train = tomato_x_train.values
x_test = tomato_x_test.values

## Compute the PCA for visualization purposes of x_train data
# Define the model with the number of components
pca = PCA(n_components=2)
# Fit the pca model to the train data
x_train_pca = pca.fit(x_train)

# Transform both train and test data
x_train_pca = pca.transform(x_train)
x_test_pca = pca.transform(x_test)

# x_train_pca and x_test_pca now contain the PCA projections of the original data

## print the explained variance ratios of each principal component
print('PCA explained variance ratios:', pca.explained_variance_ratio_)

## Plot PCA results
plt.figure(figsize=(6,5))
plt.scatter(x_train_pca[:,0], x_train_pca[:,1], c='gray', s=20, alpha=0.6, label='Train')
plt.scatter(x_test_pca[:,0], x_test_pca[:,1], facecolors='none', edgecolors='r', s=40, label='Test')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('PCA of Training and Test Data')
plt.legend()
plt.grid(alpha=0.33)
plt.show()

In [ ]:
# Hotelling T^2 and Q-residual diagnostics for the PCA model
alpha = 0.95

# Use the PCA scores to compute Hotelling T^2 and Q-residuals for both train and test sets
train_scores = x_train_pca
test_scores = x_test_pca

# Reconstruct the original data from the PCA scores
train_reconstructed = pca.inverse_transform(train_scores)
test_reconstructed = pca.inverse_transform(test_scores)

# Hotelling T^2 using the retained PCA score variances
t2_train = np.sum((train_scores ** 2) / pca.explained_variance_, axis=1)
t2_test = np.sum((test_scores ** 2) / pca.explained_variance_, axis=1)

# Q-residuals (also called SPE) from the reconstruction error
q_train = np.sum((x_train - train_reconstructed) ** 2, axis=1)
q_test = np.sum((x_test - test_reconstructed) ** 2, axis=1)

# Empirical limits estimated from the training set
# This keeps the diagnostic consistent even when only a few PCs are retained.
t2_limit = np.quantile(t2_train, alpha)
q_limit = np.quantile(q_train, alpha)

# Define outliers based on the computed limits
train_outliers = (t2_train > t2_limit) | (q_train > q_limit)
test_outliers = (t2_test > t2_limit) | (q_test > q_limit)

## Plot the Hotelling T^2 and Q-residuals for both train and test sets
plt.figure(figsize=(8, 5))
plt.scatter(t2_train, q_train, c='gray', s=24, alpha=0.7, label='Train')
plt.scatter(t2_test, q_test, facecolors='none', edgecolors='r', s=42, label='Test')
plt.axvline(t2_limit, color='tab:blue', linestyle='--', linewidth=1, label=f'T$^2$ {int(alpha*100)}% limit')
plt.axhline(q_limit, color='tab:orange', linestyle='--', linewidth=1, label=f'Q {int(alpha*100)}% limit')
plt.xlabel('Hotelling T$^2$')
plt.ylabel('Q residual')
plt.title('PCA outlier map')
plt.grid(alpha=0.3)
plt.legend(loc='best')

plt.tight_layout()
plt.show()

## show the indices of the outliers in both train and test sets
print(f'Train outliers ({train_outliers.sum()}):', np.where(train_outliers)[0].tolist())
print(f'Test outliers ({test_outliers.sum()}):', np.where(test_outliers)[0].tolist())

Afterwards we could simply remove these outliers before proceeding with the modelling steps.

## 4) PLS modelling

We implement a **PLS regression model as a baseline for comparison** with the CNN models. The PLS model is optimized for preprocessing and number of latent variables (LV) using cross-validation.

A previous study found that SNV (standard normal variate) and first-order Savitzky–Golay derivative preprocessing are the optimal choices on these datasets for PLS .
Using these preprocessing methods, we can perform a grid search to find the optimal number of latent variables for each dataset.

<code>
Tecator_moisture         --> SNV <br>
9-NIR_tomato4_ssc        --> SNV <br>          
16-CEOT_pears_2019_brix  --> SG_1st_der_w9  
</code>


In [ ]:
## Compute SNV for the Tecator dataset and plot the results
tecator_x_train_snv = snv(tecator_x_train.values)
tecator_x_test_snv = snv(tecator_x_test.values)

plt.figure(figsize=(12,3))
plt.subplot(1, 2, 1)
plt.title('Tecator SNV (train)')
plt.plot(tecator_w, tecator_x_train_snv.T)
plt.ylabel('Reflectance')
plt.xlabel('Wavelength (nm)')
plt.subplot(1, 2, 2)
plt.title('Tecator SNV (test)')
plt.plot(tecator_w, tecator_x_test_snv.T)
plt.ylabel('Reflectance)')
plt.xlabel('Wavelength (nm)')
plt.tight_layout()
plt.show()

## Compute SNV for the Tomato 
tomato_x_train_snv = snv(tomato_x_train.values)
tomato_x_test_snv = snv(tomato_x_test.values)

#### [IMPLEMENT PLOT]

## Compute Savitzky-Golay 1st derivative for the pears dataset
pear_x_train_1d = savgol_filter(pear_x_train.values, window_length=9, polyorder=2, deriv=1)
pear_x_test_1d = savgol_filter(pear_x_test.values, window_length=9, polyorder=2, deriv=1)

#### [IMPLEMENT PLOT]


We optimise a PLS model: cross-validation selects the best number of latent variables, then we compute/plot performance metrics for both train and test data using that optimal configuration.

**CVAR** is the coefficient of variation of the prediction error, computed as $$\mathrm{CVAR} = 100 \times \frac{\mathrm{RMSE}}{\mathrm{mean}(y_{\text{true}})}$$ It expresses the RMSE as a percentage of the mean reference value, so lower values indicate better predictive accuracy.

**SDR** is the standard deviation ratio, computed as $$\mathrm{SDR} = \frac{\mathrm{std}(y_{\text{true}})}{\mathrm{RMSE}}$$  It measures how large the natural variation of the reference values is compared with the prediction error. Higher SDR values indicate better model performance.


In [ ]:
## Optimize PLS for the Tecator SNV dataset using the predifined function pls_optimization_cv_stop2().
# best_LV, CV_RMSE = pls_optimization_cv_stop2(tecator_x_train_snv, tecator_y_train, nmax=20, plot_opt=True, stop_criteria=0.001)
# print('Chosen number of LV:', best_LV, '\nCV RMSE:', CV_RMSE)
# ## Plot the PLS regression diagnostics and metrics using the predifined function pls_prediction_metrics2().
# f = pls_prediction_metrics2(tecator_w, tecator_x_train_snv, tecator_y_train, tecator_x_test_snv, tecator_y_test,
#                             'SNV','Moisture', lv=best_LV,
#                             plot_pred=True, plot_vip=False)


#### [COMMENT/UNCOMENT THE DIFFERENT SECTIONS AND RUN IT TO SEE THE RESULTS]

## Optimize PLS for the Tomato SNV dataset
# best_LV, CV_RMSE = pls_optimization_cv_stop2(tomato_x_train_snv, tomato_y_train, nmax=20, plot_opt=True, stop_criteria=0.001)
# print('Chosen number of LV:', best_LV, '\nCV RMSE:', CV_RMSE)

# f = pls_prediction_metrics2(tomato_w, tomato_x_train_snv, tomato_y_train, tomato_x_test_snv, tomato_y_test,
#                             'SNV','SSC', lv=best_LV,
#                             plot_pred=True, plot_vip=False)


### Optimize PLS for the Pear 1st deriv dataset
best_LV, CV_RMSE = pls_optimization_cv_stop2(pear_x_train_1d, pear_y_train, nmax=20, plot_opt=True, stop_criteria=0.001)
print('Chosen number of LV:', best_LV, '\nCV RMSE:', CV_RMSE)

f = pls_prediction_metrics2(pear_w, pear_x_train_1d, pear_y_train, pear_x_test_1d, pear_y_test,
                            '1st deriv','SSC', lv=best_LV,
                            plot_pred=True, plot_vip=False)

After fixing the latent variables, this cell fits the final PLS model and plots how much variance each component explains so we understand the contribution of successive factors.


In [ ]:
pls2 = PLSRegression(n_components=best_LV)
pls2.fit(tecator_x_train_snv, tecator_y_train)

## We compute the explained variance using the predifined function pls_explained_variance().
pls_explained_variance(pls2, tecator_x_train_snv, tecator_y_train)

## 5) CNN models for regression

In this section we define two slightly different (simple) CNN architectures.

Utility functions for feature scaling are defined so that we always standardise test data with statistics computed on the corresponding training subset, respecting the validation protocol.


In [ ]:
## Since the test set is unknown (we are not suppose to have access to it during the
## optimization of the model) the scaling process should take this into account. We
## have to define a scaler based only on the train data, and apply it to the test data.

def standardize_column(X_train, X_test):
    ## We train the scaler on the full train set and apply it to the test set
    scaler = StandardScaler().fit(X_train)
    ## for columns we fit the scaler to the train set and apply it to the test set
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return [X_train_scaled, X_test_scaled]

Column-wise standardisation is applied to the spectra and the standardised curves are plotted, showing how the preprocessing centres and scales each wavelength. This type of preprocessing is very useful for Neural Networks, as it helps to stabilise the training process and improve convergence.

**Note**: we do not perform any type of preprocessing optimization like we did for PLS. These CNN models will use the raw spectra (standardized column-wise) as input, and the preprocessing is fixed for all experiments. In many occasion, it might be useful to optimize the preprocessing for the CNNs as well (especially for shallow architectures), but we will not do that in this tutorial. 

In [ ]:
## Standardize on columns
tecator_x_train_scaled, tecator_x_test_scaled = standardize_column(tecator_x_train, tecator_x_test)


## plot the standardized test spectra of each individual fruit
plt.figure(figsize=(12,3))
plt.subplot(121)
plt.title('train')
plt.plot(tecator_w, tecator_x_test_scaled.T,  alpha=0.7)
plt.ylabel('Standardized absorbance', fontsize=10)
plt.xlabel('Wavelength (nm)', fontsize=10)
plt.axhline(0,c='k')
plt.subplot(122)
plt.title('test')
plt.plot(tecator_w, tecator_x_test_scaled.T, alpha=0.7)
plt.ylabel('Standardized absorbance', fontsize=10)
plt.xlabel('Wavelength (nm)', fontsize=10)
plt.axhline(0,c='k')
plt.tight_layout()
plt.show()

## Do the same for the other two datasets (tomato and pear)
tomato_x_train_scaled, tomato_x_test_scaled = standardize_column(tomato_x_train.values, tomato_x_test.values)
pear_x_train_scaled, pear_x_test_scaled = standardize_column(pear_x_train.values, pear_x_test.values)


### 5.1) Implementing the CNN architecture

Lets create a simple CNN architecture composed of **1 convolutional layer with just 1 filter, 1 dense layer and a final output layer** (for regression purposes). We will also add L2 regularization on all layers for helping stabilize the learning process and decrease overfitting problems.
The adjustable hyperparameters will be the width of the convolutional filter, the number of units in the dense layer and the strength of the L2 regularization.
If we use multiple convolutional filters, the number of extracted features upon flattening will increase and the number of parameters in the dense layers will also increase. In this situation one can consider decreasing the number of features by using a pooling layer after the last convolutional layer.
The implementation of this model in tensorflow/keras is straightforward.<br>
Some theoretical remarks before the implementation:
<br>
- **Weights initialization:**  "*The weights are initialized by a zero-mean Gaussian distribution whose standard deviation is $\sqrt{\frac{2}{n_i}}$, where $n_i$ is the number of input neurons*". This corresponds to <code>he_normal()</code> initializer in tf.keras. We set the flag "seed" to a fixed value for reproducibility purposes.
- **Loss function:** The total loss function is $Loss = MSE + \frac{1}{2} \lambda \sum{w_i^2}$, i.e. the mean squared error (MSE) plus an L2 penalty $\lambda$ on the weights $w_i$. Using Keras this is done by choosing the loss function as <code>'mse'</code> and apply an L2 regularization to the weights (by using <code> tf.keras.regularizers.l2(beta))</code>. For simplification purposes we use a global L2 regularization, i.e., the same penalty for the weights in all the layers.
- **Optimizer:** We use the 'Adam' optimizer here, due to its proven record of good performance.

We build `create_model_cnn`, a compact functional CNN that reshapes the spectra, applies a single 1D convolution, and finishes with a dense regression head—this is the architecture used in the introductory CNN experiments.



We implement two Keras builders (`create_model_1` and `create_model_2`) that generate 1D CNN regression models, exposing the hyperparameters (dense layers, filter size, regularisation) needed for later optimisation experiments. The first model is a simple CNN with one convolutional layer, one dense layer, and an output layer. The second model is a more complex CNN that allows for multiple dense layers (during NAS) and adds dropout for extra regularization to help prevent overfitting.

In [ ]:
## Run this function to make the computations reproducible
reproducible_comp()

## Define the model
## Instead of using the functional API, for this CNN we could use a "Sequential model" to create the model.

# def create_model_cnn1(input_dims, filter_size, dense_units, reg_beta, Y_mean):
def create_model_cnn1(input_dims, filter_size, dense_units, reg_beta):
    ## Layers dimensions
    INPUT_DIMS = input_dims   # lenght of the input vector: np.shape(x_train)[1]
    ## number of convolutional kernels/filters
    K_NUMBER = 1
    ## kernel/filter size
    K_WIDTH = filter_size
    ## kernel/filter stride (step)
    K_STRIDE = 1
    ## number of units in the dense layer
    FC_UNITS = dense_units
    ## Output dimensions, 1 for regression
    REG_OUTPUT_DIMS = 1
    ## Global (all layers) L2 regularizer parameter
    K_REG = tf.keras.regularizers.L2(reg_beta)
    ## Weights initialization for multiple layers. Fixing a seed for reproducibility
    K_INIT = tf.keras.initializers.he_normal(seed=123)

    ####### Architecture of the main model ##############
    ## Input Layer
    inputs = layers.Input(shape=(INPUT_DIMS,), name='INPUT')
    ## Reshape the input to 1D and to accommodate for batch size
    x = layers.Reshape((INPUT_DIMS, 1),name='RESHAPE')(inputs)
    ## Convolutional layer
    x = layers.Conv1D(filters=K_NUMBER,
                      kernel_size=K_WIDTH,
                      strides=K_STRIDE,
                      padding='same',
                      kernel_initializer=K_INIT,
                      kernel_regularizer=K_REG,
                      bias_regularizer=K_REG,
                      activation=keras.layers.LeakyReLU(negative_slope=0.2),
                      name='CONVOLUTIONAL')(x)
    ## Pooling layer
    # x = layers.AveragePooling1D(pool_size=5, name='POOLING')(x) ## reduces the number of features by a factor of 5 

    ## Add flatten layer for reshaping the dimensions
    x = layers.Flatten(name='FLATTEN')(x)
    ## Fully connected layer
    x = layers.Dense(FC_UNITS,
                     kernel_initializer=K_INIT,
                     kernel_regularizer=K_REG,
                     bias_regularizer=K_REG,
                     activation=keras.layers.LeakyReLU(negative_slope=0.2),
                     name='DENSE')(x)
    ## Regression output
    reg_output = layers.Dense(REG_OUTPUT_DIMS,
                              kernel_initializer=K_INIT,
                              kernel_regularizer=K_REG,
                              bias_regularizer=K_REG,
                              # bias_initializer=keras.initializers.Constant(Y_mean),
                              activation='linear',
                              name='REG_OUTPUT')(x)

    # Create the model with multiple outputs
    model_cnn1 = Model(inputs=inputs, outputs=[ reg_output], name='MODEL_CNN1')
    
    return model_cnn1




#### Create model_2 function is similar to model_1 but uses a different implementation of the layers.
def create_model_2(input_dims,num_FC_layers, num_FC_units, filter_size, DROPOUT, reg_beta):
    ## Layers dimensions
    INPUT_DIMS = input_dims
    K_NUMBER = 1
    K_WIDTH = filter_size
    K_STRIDE = 1
    REG_OUTPUT_DIMS = 1

    ## Global (all layers) L2 regularizer parameter
    beta = reg_beta
    K_REG = tf.keras.regularizers.l2(beta)

    ## Weights initialization for multiple layers
    K_INIT = tf.keras.initializers.he_normal(seed=123)

    ## Architecture of the main model
    input_layer = layers.Input(shape=(INPUT_DIMS,), name='INPUT')
    x = layers.Reshape((INPUT_DIMS, 1),name='RESHAPE')(input_layer)
    x = layers.Conv1D(filters=K_NUMBER,
                      kernel_size=K_WIDTH,
                      strides=K_STRIDE,
                      padding='same',
                      kernel_initializer=K_INIT,
                      kernel_regularizer=K_REG,
                      activation=keras.layers.LeakyReLU(negative_slope=0.2),
                      name='CONVOLUTIONAL')(x)

    x = layers.Flatten(name='FLATTEN')(x)

    for i in range(0, num_FC_layers):
        x = layers.Dense(num_FC_units[i],
                         kernel_initializer=K_INIT,
                         kernel_regularizer=K_REG,
                         activation=keras.layers.LeakyReLU(negative_slope=0.2),
                         name='DENSE'+str(i))(x)
        if i != num_FC_layers - 1:  # Only add dropout if it's not the last iteration
            x = layers.Dropout(DROPOUT[i], name='DROPOUT'+str(i))(x)

    # Regression output
    reg_output = layers.Dense(REG_OUTPUT_DIMS,
                              kernel_initializer=K_INIT,
                              activation='linear',
                              name='REG_OUTPUT')(x)

    # Create the model with multiple outputs
    model_cnn = Model(inputs=input_layer, outputs=[reg_output], name='MODEL_CNN_V2')

    return model_cnn

Lets instantiate the model with some hyperparameters.
Using the helper above, we instantiate a concrete CNN (`cnn_1`) with 128 dense units, 7-point filters, and L2 regularisation=0.001, and print the model summary to review its layers and parameter counts.



In [ ]:
## Create a CNN using create_model_cnn1(input_dims,filter_size, dense_units, reg_beta)
cnn_1 = create_model_cnn1(np.shape(tecator_x_train)[1], 7, 128, 0.001)

# Show the summary of the model
cnn_1.summary()

This cell renders a diagram of `cnn_1`, making a visual representation of the sequence of layers, tensor shapes, and activations.


In [ ]:
## Plot the architecture
tf.keras.utils.plot_model(cnn_1,
                          show_shapes=True,
                          show_layer_activations=True,
                          show_dtype=False ,
                          show_layer_names=False,
                          rankdir='TB',
                          expand_nested=False,  dpi=64)

The next step consists in training our CNN. For that purpose we define a few useful callback functions, compile our model choosing the type of gradient optimizer (ADAM in this case) and train the model. Callbacks are function that are called during the training process at certain points (e.g., at the end of an epoch, before or after a batch, etc.). They can be used to monitor the training process, save the model, adjust learning rates, and more. In this case, we will use callbacks to implement early stopping, reduce learning rate on plateau, and save the best model.

We split the train set into calibration and validation subsets for the model training. The CNN will use the calibration data to train the model weights and the validation set for monitoring the process. This is useful to check for model overfitting. In this case will will use a callback named EarlyStopping() that will also help on this.
We print the current working directory so you know where intermediate models and outputs will be saved when you replicate the workflow.

### 5.2) Train the CNN model

The workflow for the first training experiment starts here: clear old sessions (i.e., reset the TensorFlow/Keras backend) $\rightarrow$ split the training data into calibration/validation sets $\rightarrow$ standardise the data splits $\rightarrow$ configure callbacks (early stopping, learning-rate decay, checkpoints, etc) $\rightarrow$ compile `cnn_1` $\rightarrow$ fit/train the CNN while logging progress. 


In [ ]:
## Clear model parameter that might be in memory
keras.backend.clear_session()
reproducible_comp()

## Define the data to train the model (change it here for tecator, tomato or pear datasets):
x_train = tecator_x_train.values
y_train = tecator_y_train.values
x_test = tecator_x_test.values
y_test = tecator_y_test.values
INPUT_DIMS = np.shape(x_train)[1]


#### DEFINE CALLBACKS FOR TRAINING
## EarlyStopping: stop the training if the validation loss stops improving by "min_delta" over "patience" number of epochs
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss",patience=50,restore_best_weights=True)
## ReduceLROnPlateau: Dynamicallyy reduces the learning rate by "factor" if the validation loss does not improve over "patience" epochs
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)

## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
progressbar = TqdmCallback()
## Alternatively, we can monitor the training in real time using PlotLossesKerasTF (it is a bit slower)
# liveplot = PlotLossesKerasTF()

## Save the best model based on the val loss (the val loss is not used at any point during training)
model_name = 'cnn1.keras'
checkpointer = ModelCheckpoint(filepath=model_name, monitor='loss', verbose=0, save_best_only=True)

#### CALIBRATION AND VALIDATION SPLIT
## Split train data into calibration and validation sets (even better if we use CV instead of a single split)
## First we split the train into calibration and validation sets.
x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train, y_train, test_size=0.15, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test)
_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)

print('Train calibration set shape:', x_train_cal_scaled.shape)
print('Train validation set shape:', x_train_val_scaled.shape)
print('Test set shape:', x_test_scaled.shape)


##### DEFINE TRAINING HYPERPARAMETERS
## Number of samples in each batch
BATCH_SIZE=64
## Learning rate (this value can/should be tuned)
LR= 0.01
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=200

######### Define the model hyperparameters 
# (LR=0.01, FC_UNITS=92, FILTER_SIZE=35, L2_REG=0.03 -> test RMSE ~1.887)
# (LR=0.00494, FC_UNITS=256, FILTER_SIZE=25, L2_REG=3.11e-8 -> test RMSE ~1.57)
FC_UNITS = 128
FILTER_SIZE = 7
L2_REG = 0.001


## Create the model
# Y_mean = np.mean(y_train_cal)
cnn_1 = create_model_cnn1(INPUT_DIMS, FILTER_SIZE, FC_UNITS, L2_REG)
## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
cnn_1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=[keras.metrics.RootMeanSquaredError(name="rmse")])


## Train the model and visualize the training process
##++++++++++++ TRIAL 1 ++++++++++++++++++++++++++++++
#h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
#                   validation_data = (x_train_val_scaled, y_train_val) ,
#                   callbacks=[early_stop, rdlr, checkpointer, liveplot], verbose=0)

##++++++++++++ TRIAL 2 ++++++++++++++++++++++++++++++
## Alternatively, use progressbar to visualize the train. Pass the training into a history object "h1" for later use
h1 = cnn_1.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
                   validation_data = (x_train_val_scaled, y_train_val) ,
                   callbacks=[early_stop,rdlr,progressbar,checkpointer], verbose=0)

print(f'\n Training completed... \n Loading best model weights from {model_name}...')

## After the model finishes the training, load the best model weights
cnn_1.load_weights(model_name)

# Clear session to free up resources and avoid clutter from old models/layers
keras.backend.clear_session()

Now we visualize the training history and compute error metrics on the calibration, validation and test sets.
Using the fitted `cnn_1`, we plot training/validation loss curves, compute R² and RMSE on calibration, validation, and test sets, and clear the Keras backend to free resources.



In [ ]:
## Take a look at the training process by plotting the models history.
plt.figure(figsize=(6,3))
plt.plot(h1.history['loss'], label='Train loss')
plt.plot(h1.history['val_loss'], label='Val loss')
plt.yscale('log')
plt.ylabel('Loss')
plt.xlabel('Epochs')
# plt.ylim(0.5,1)
plt.legend()
## In case you used ReduceLROnPlateau() you can plot the lr as well
## [UNCOMMENT THE FOLLOWING LINES IF YOU USED ReduceLROnPlateau() CALLBACK]
ax2 = plt.gca().twinx()
ax2.plot(h1.history['learning_rate'], color='r', ls='--')
ax2.set_ylabel('learning rate',color='r')
plt.tight_layout()
plt.show()


## Compute RMSE metrics for TRAIN and TEST sets
y_train_cal_pred = cnn_1.predict(x_train_cal_scaled)
y_train_val_pred = cnn_1.predict(x_train_val_scaled)
y_test_pred = cnn_1.predict(x_test_scaled)

## Compute train error scores
R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
R2_train_val = r2_score(y_train_val, y_train_val_pred)
rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)

## Compute test error scores
R2_test = r2_score(y_test, y_test_pred)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )

## Clear clutter from previous session
# keras.backend.clear_session()
# print('\n Keras backend cleared...')

<div class="alert alert-block alert-warning">
<b>Suggestion:</b> Try running the previous cells by commenting section "TRIAL 2" and uncomment "TRIAL 1". <br>
<b>Suggestion:</b> Try modifying the input data, or changing one of the hyperparameter by hand (filter size, dense units, L2 regularization) and see how the performance changes.<br>
<b>Suggestion:</b> In the training cell uncomment the rdlr = ReduceLROnPlateau line and add "rdlr," to the callbacks list. You can see the behaviour of the chaning learning rate in the history plot by uncommenting the lines related to ax[2].<br>
</div>

We call `plot_prediction2` to visualise measured versus predicted values on the test set (validation plot is left commented in case you want to enable it).

<div class="alert-success ">
<b>DÁRIO:</b> Nesta parte podemos mostrar o impacto de fazer a inialização do bias do output layer com a média do target. A ideia é que o modelo comece a aprender a partir de um ponto mais próximo da solução final, e não de um valor aleatório. Isso pode ajudar a acelerar a convergência e melhorar a performance do modelo, especialmente em datasets pequenos ou com pouca variabilidade.
</div>

In [ ]:
## Plot validation predictions (lathough the name TEST appears by default as the name of the right column of info)
# plot_prediction2(y_train_cal, y_train_val, y_train_cal_pred, y_train_val_pred, 'CNN 1 (val)', savefig=False, figname=None)

## Plot test predictions
plot_prediction2(y_train_cal, y_test, y_train_cal_pred, y_test_pred, 'CNN 1 (test)', savefig=False, figname=None)

### 5.2.1) Finding a good learning rate range for the CNN model

The LR range test measures how quickly an untrained model begins learning as the learning rate increases. LRFinder briefly trains a freshly initialized model while exponentially increasing the learning rate after every batch and recording the corresponding smoothed training loss. The resulting curve helps select a learning rate where the loss decreases rapidly but remains safely below the point at which training becomes unstable or diverges.

In [ ]:
## The range test must start from fresh weights and use a disposable model.
## Define the start and end learning rates, the number of steps, and the batch size for the range test.
LR_START = 1e-7
LR_END = 0.25
LR_STEPS = 300
LR_BATCH_SIZE = 64

steps_per_epoch = int(np.ceil(len(x_train_cal_scaled) / LR_BATCH_SIZE))
lr_epochs = 2*int(np.ceil(LR_STEPS / steps_per_epoch))
print(f"Optimizing LR for at most {LR_STEPS} steps ({lr_epochs} epochs)")

## Instantiate a new model (in this case with the same architecture as cnn_1)
reproducible_comp()
cnn_lr_finder = create_model_cnn1(INPUT_DIMS, FILTER_SIZE, FC_UNITS, L2_REG)
cnn_lr_finder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR_START),  loss="mse",)

## Perform the learning rate range test
lr_finder = LRFinder( start_lr=LR_START, end_lr=LR_END,  max_steps=LR_STEPS,  smoothing=0.99,)

cnn_lr_finder.fit( x_train_cal_scaled, y_train_cal,
    batch_size=LR_BATCH_SIZE, epochs=lr_epochs,
    shuffle=True, callbacks=[lr_finder],  verbose=0,)

print(
    f"Collected {len(lr_finder.lrs)} batches: "
    f"{lr_finder.lrs[0]:.2e} to {lr_finder.lrs[-1]:.2e}"
)
lr_suggestions = lr_finder.suggestions()
print(lr_suggestions)
lr_finder.plot(
    skip_end=0,
    steepest_lr=lr_suggestions["steepest_descent"],
)
plt.show()

# Never continue training the model used for the range test.
del cnn_lr_finder

A suitable learning rate lies on the steeply descending part of the LR-finder curve, before the loss reaches its minimum and becomes unstable. Here, the heuristics agree on approximately $10^{-2}$, so an LR between 0.01 and 0.02 is reasonable, with 0.01 providing the safer starting point.

The learning rate that we used previously (0.01) was already a good choice and we will stick to it for next experiments.

### 5.3) Grid search for one hyperparameter...
A manual grid search over different L2 regularisation strengths is executed here. For each value we retrain `cnn_1`, record performance metrics, and collect the results into a dataframe for later analysis.


In [ ]:
######### The input data ##########
## We already have the train and test datasets defined in the previous section. We will use the same datasets for the grid search.
## Otherwise uncomment the following lines to define the datasets for the grid search
# x_train = tecator_x_train.values
# y_train = tecator_y_train.values
# x_test = tecator_x_test.values
# y_test = tecator_y_test.values
######### Train test split ########
# x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train, y_train, test_size=0.2, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
# x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test)
# _, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)



################## Define some of the training hyperparameters
## Number of samples in each batch
BATCH_SIZE=64
## Learning rate 
LR=0.01
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=200

## Define the fixed model hyperparameters
FC_UNITS = 128
FILTER_SIZE = 7

## Hyperparameter ranges to explore
# filter_sizes = [5,7,9,15,25]
# n_units = [64, 96, 128, 256]
l2_reguls = [0.1, 0.01, 0.001, 0.0001, 0.0]

metrics = [] ## empty list to store the error metrics

################# Start of the training loop for the grid search
for L2_REG in l2_reguls:
    ## Clear model parameter that might be in memory
    keras.backend.clear_session()
    reproducible_comp()

    ########### Callbacks to use during training (defined inside the loop to make sure they are reset for each training session) #######
    ## EarlyStopping: stop the training if the validation loss stops improving by "min_delta" over "patience" number of epochs
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss",patience=50,restore_best_weights=True)
    ## ReduceLROnPlateau: Dynamicallyy reduces the learning rate by "factor" if the validation loss does not improve over "patience" epochs
    rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
    ## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
    progressbar = TqdmCallback()
    ## Save the best model based on the val loss (the val loss is not used at any point during training)
    ## We are saving each model with a different name based on the L2 regularization parameter to avoid overwriting the previous models
    ## but here we could just use the same name and overwrite the previous model since we are loading the best weights after each training session.
    model_name = f'/content/L2_models/cnn_1_l2={L2_REG}.keras'
    checkpointer = ModelCheckpoint(filepath=model_name, monitor='val_loss', verbose=0, save_best_only=True)

    ## Create the model. It is important to create a new model instance each time to make sure the weights are reset
    model = create_model_cnn1(INPUT_DIMS, FILTER_SIZE, FC_UNITS, L2_REG)
    ## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])

    ## train the model
    model.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
              validation_data = (x_train_val_scaled, y_train_val),
              callbacks=[early_stop, checkpointer, rdlr, progressbar], verbose=0)

    print(f'\n Training completed... \n Loading best model weights from {model_name}...')
    ## Load the best model weights
    model.load_weights(model_name)

    ## Compute RMSE metrics for TRAIN and TEST sets
    y_train_cal_pred = model.predict(x_train_cal_scaled)
    y_train_val_pred = model.predict(x_train_val_scaled)
    y_test_pred = model.predict(x_test_scaled)

    ## Compute train error scores
    R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
    rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
    R2_train_val = r2_score(y_train_val, y_train_val_pred)
    rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)

    ## Compute test error scores
    R2_test = r2_score(y_test, y_test_pred)
    rmse_test = root_mean_squared_error(y_test, y_test_pred)

    print('\n----------------------------')
    print(f'CNN with l2_reg = {L2_REG}')
    print('----------------------------')
    print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
    print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
    print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )
    ## Append the metrics to the list
    metrics.append([L2_REG, rmse_train_cal, rmse_train_val, rmse_test, R2_train_cal, R2_train_val, R2_test])


    ## Clear clutter from previous session
    # keras.backend.clear_session()
    # print('\n Keras backend cleared...')
############### END OF GRID SEARCH LOOP ####################


## Convert the metrics list to a dataframe
metrics_df = pd.DataFrame(metrics, columns=['l2_reg', 'RMSE_train_cal', 'RMSE_train_val', 'RMSE_test', 'R2_train_cal', 'R2_train_val', 'R2_test'])

## Save the metrics to a csv file
# metrics_df.to_csv('cnn_1_metrics.csv', index=False)

Now that the grid search is over lets take a look at the results stored in our dataframe "metrics_df"
The metrics gathered during the grid search are displayed so that you can identify which L2 penalty delivers the best validation behaviour.



In [ ]:
## Display the dataframe to check the optimization results.
metrics_df.round(4) ## rounding the values to 3 decimal places

By looking into the validation set metrics what is the best L2 regularization to use?

We only computed the RMSE of the test set for comparison purposes. **Remember, all decisions related with the model's hyperparameters should be based on the validation metrics (also known as tuning metrics)**. The test set metrics should only be computed after choosing the best model.

One of the problems related to this way of doing hyperparameter optimization is that if we change one of the fixed hyperparameters (e.g., filter size, number of dense units) we will have to repeat the whole grid search for L2 regularization. This is a very time-consuming process and it is not feasible to do it manually. In the next section we will implement an automatic hyperparameter optimization procedure using Optuna.

### 5.4) Bayesian Optimization (BO) of the hyperparameters... 
In the previous grid search example, we optimized one hyperparameter at a time. Usually, neural network hyperparameters are correlated, so when we change one, the whole model changes and that will affect the role of the other hyperparamters. One of the best way to do this search is by looking into many combinations of hyperparameters and see which combination provides the best results. Since this can result in a combinatorial explosion of possibilities, we can use clever methods of search such as genetic algorithms,  Bayesian algorithms etc. In this case we will make use of a library called Optuna that has several optimization algorithms available for optimizing our model's hyperparameters.

#### 5.4.1) Using a single validation split strategy

- for simplification we use a fixed single validation split per trial.
- We use a TPE (Tree Parzen Estimator) algorithm (a type of bayesian search). We can use prunning to stop trials that are not promising.
- We not only optimize the hyperparameters of the layers, but also the architecture (by implementing flags that create layers as needed, within bounds). So, in fact we are performing a joint HPO and NAS (Neural Architecture Search)

**Step 1)** Create an objective function for the "optimizer to optimize". In this case the objective function will be the validation split RMSE.
This cell defines the Optuna objective used for Bayesian optimisation: it standardises the data, samples CNN hyperparameters (dense layout, filter size, dropout, L2), performs model training (with some callbacks), and returns the validation RMSE.

In [ ]:
## We might need this extra library for implementing prunning during the hyperparameter optimization.
from optuna.integration import TFKerasPruningCallback


## Set the path where the computed models are saved
## Beware of path notation differences between windows (\\) and linux (\) systems
LOCAL_MODELS_DIR = "/content/local_models"
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
path = f"{LOCAL_MODELS_DIR}/"


## Define the data to train the model (change it here for tecator, tomato or pear datasets):
x_train = tecator_x_train.values
y_train = tecator_y_train.values
x_test = tecator_x_test.values
y_test = tecator_y_test.values
INPUT_DIMS = np.shape(x_train)[1]
#### CALIBRATION AND VALIDATION SPLIT
## Split train data into calibration and validation sets (even better if we use CV instead of a single split)
## First we split the train into calibration and validation sets.
x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train, y_train, test_size=0.15, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test)
_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)


## initialize an empty list for storing additional metrics
# metricas = []


## Standardize the train and test sets
x_train_scaled, x_test_scaled = standardize_column(x_train, x_test)


## Define the objective function to monitor during optimization, set the hyperparameters ranges, etc.
def objective(trial):

    print('\n\n-------------- TRIAL NUMBER: ',trial.number,'------------------')

    ## Clear clutter from previous session
    keras.backend.clear_session()
    reproducible_comp()

    #### Callbacks to use during training
    early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
    ## Note that the min_lr in rdlr is the one found in the LRFinder test
    rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
    ## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
    progressbar = TqdmCallback()
    ## define the model/study name. In this case we will save a model with the same name of its trial
    MODEL_NAME=f'study1_cnn_trial={trial.number}.keras'
    checkpointer = ModelCheckpoint(filepath=path+MODEL_NAME, monitor='val_loss', verbose=0, save_best_only=True)


    ## In the follow steps we define the search space for each of the hyperparameters. This is done by using
    ## distributions/intervals of integers or floats from a min, to a max with a certain step.
    ##
    ## Number of FC layers (1 to 3 layers)
    NUM_FC_LAYERS = trial.suggest_int("num_FC_layers",1,3, step=1)
    ## Number of units per layer
    NUM_FC_UNITS = [int(trial.suggest_int("num_FC_UNITS_"+str(i), 32, 256, step=16)) for i in range(0,NUM_FC_LAYERS)]
    ## Filter size  (K_WIDTH)
    FILTER_SIZE = int(trial.suggest_int("filter_size", 5, 75, step=5))
    ## Dropout rate (DROPOUT)
    DROPOUT_RATE = [trial.suggest_float("DROPOUT_"+str(i), 0., 0.5, step=0.005) for i in range(0,NUM_FC_LAYERS-1)]
    ## L2 regularization
    REG_BETA = trial.suggest_float("reg_beta", 0, 0.05, step=0.001)
    ## We can add batch size to the hyperparameters to be optimized. This hyperparameter is not used directly by the model()
    ## but is used in the training phase.
    # BATCH_SIZE = int(trial.suggest_int("batch_size", 32, 256, step=32))
    ## In this case we keep if fixed as before for simplicity. The same goes for the learning rate LR.
    BATCH_SIZE=64
    ## Learning rate (this is a heuristic value, it should be tuned)
    LR=0.01    #*BATCH_SIZE/256.
    EPOCHS=300

    ## Instantiate the model for a new set of hyperparameters. Use use create_model_2() that is similar to create_model_cnn()
    ## but uses a different implementation of the layers. Instead of using a Sequential model, we use the functional API to create the model.
    ## This way we can use the same model for different hyperparameters
    model = create_model_2(INPUT_DIMS, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, REG_BETA)
    print('\n\ncreate_model(',INPUT_DIMS, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, REG_BETA,')')
    # model.summary()
    print('\n')

    ## Compile the model, use Adam() as optimizer, and mean squared error (mse) as loss function
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])

    ## Define the prunning callback to stop the training and skip to next trial.
    pruning_callback = TFKerasPruningCallback(trial,monitor="val_mse",)
    
    ## Train the model on train_cal data and validate it on train_val data
    model.fit(x_train_cal_scaled, y_train_cal, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,\
                  validation_data = (x_train_val_scaled, y_train_val),\
                  callbacks=[rdlr, early_stop, checkpointer, progressbar, pruning_callback],\
                  verbose=0)

    ## Load the best model weights
    model.load_weights(path+MODEL_NAME)

    ## Compute auxiliary metrics
    ## Note: model.evaluate() returns a list of metrics, the first one is the loss (mse) 
    scores_cal = np.sqrt(model.evaluate(x_train_cal_scaled, y_train_cal, verbose=0)[1])
    print('Calibration RMSE = {}'.format(scores_cal))

    scores_val = np.sqrt(model.evaluate(x_train_val_scaled, y_train_val, verbose=0)[1])
    print('Validation RMSE = {}'.format(scores_val))

    ## Clear clutter from previous session
    keras.backend.clear_session()

    ## print end of trial message
    print('\n\n-------------- END OF TRIAL NUMBER: ',trial.number,'------------------\n\n\n\n')

    ## The metric we want to minimize. In this case we are chosing validation RMSE as the metric to minimize.
    return scores_val

**Step 2)** Since we have defined the objective function, we can now run the optimization procedure. The optimization will be based on the validation RMSE, and we will run a number of trials to explore different hyperparameter combinations. We need to define the type of optimizer and the number of trials to run.

In [ ]:
## Enable Optuna logger (for tracking purposes)
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))

## We only need to run this cell once for Optuna to log the trials

In [ ]:
## Unique identifier of the study
STUDY_NAME = "study_1"

## Write the live DB to local disk
LOCAL_DB_DIR = "/content/optuna_dbs"
os.makedirs(LOCAL_DB_DIR, exist_ok=True)
LOCAL_DB_PATH = f"{LOCAL_DB_DIR}/{STUDY_NAME}.db"

STORAGE_NAME = f"sqlite:///{LOCAL_DB_PATH}"

## Create a study that will, in this case minimize the objective function previously defined
## We use TPE with a cte seed (for reproducibility), consider_endpoints = True (to consider the min and max values of the search space).
study_1 = optuna.create_study(study_name=STUDY_NAME, 
                              storage=STORAGE_NAME, 
                              direction='minimize', 
                              sampler = optuna.samplers.TPESampler(seed = 123,
                                                                   consider_endpoints = True,
                                                                   multivariate = True,
                                                                   n_startup_trials = 55,
                                                                   warn_independent_sampling=False),
                              pruner = optuna.pruners.SuccessiveHalvingPruner(min_resource=75, 
                                                                             reduction_factor=3, 
                                                                             min_early_stopping_rate=0),
                              load_if_exists=True)

## Add a hyperparameter set that we that works more or less well to the study
## This is useful to start the optimization from a good point
study_1.enqueue_trial({"num_FC_layers":1,
                      "num_FC_UNITS_0":128,
                      "filter_size":7,
                      "reg_beta":0.001
                      })


## Start optimization with a budget of n_trials
study_1.optimize(objective, n_trials = 200, show_progress_bar=True)

**Step 3)** Lets inspect the results of the optimization procedure. The best trial is printed, showing the hyperparameters that yielded the lowest validation RMSE.

For convinience we have precomputed a study with 400 trials for the Tecator dataset. You can load the precomputed study (or your study) in the next cell and inspect the results. 

In [ ]:
## loat the precomputed study_1
DB_PATH_PRECOMPUTED = f"{data_path}/models/precomputed/study_1.db"
study_1 = optuna.load_study(study_name="study_1", storage=f"sqlite:///{DB_PATH_PRECOMPUTED}")

## load your study (if it was not loaded before)
# study_1 = optuna.load_study(study_name="study_1", storage=f"sqlite:///{DB_PATH}")

print('Best trial:',study_1.best_trial.number)
print('Best trial value:',study_1.best_trial.value)
print('Best trial hyperparameters:', study_1.best_trial.params)

**Step 4)** Here we load the best presaved model (from the HPO) and predict the test set. We compare that with a full model retrain using the same hyperparameters but the full training set (calibration + validation). Sometimes this last step improves the final metrics because more data is used for training, but sometimes it does not!

In [ ]:
path=f"{data_path}/models/"
best_trial_number = study_1.best_trial.number

## Load precomputed best model 
best_model = tf.keras.models.load_model(path+'precomputed/study1_cnn_trial='+f'{best_trial_number}.keras')

## Load the model from the best trial saved during the HPO
# best_model = tf.keras.models.load_model(path+f'study1_cnn_trial={best_trial_number}.keras')

## Make model predictions
y_train_cal_pred = best_model.predict(x_train_cal_scaled)
y_train_val_pred = best_model.predict(x_train_val_scaled)
y_test_pred = best_model.predict(x_test_scaled)

## Use prediction metrics to compute the RMSE and R2 scores
R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
R2_train_val = r2_score(y_train_val, y_train_val_pred)
rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)
R2_test = r2_score(y_test, y_test_pred)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print('\n----------------------------')
print('----------------------------')
print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )

Train the best model from scratch using the full training set.

In [ ]:
keras.backend.clear_session()
reproducible_comp()

path="/content/local_models"

## The input data
## First we split the train into calibration and validation sets.
## Define the data to train the model (change it here for tecator, tomato or pear datasets):
x_train = tecator_x_train.values
y_train = tecator_y_train.values
x_test = tecator_x_test.values
y_test = tecator_y_test.values
INPUT_DIMS = np.shape(x_train)[1]
#### CALIBRATION AND VALIDATION SPLIT
## Split train data into calibration and validation sets (even better if we use CV instead of a single split)
## First we split the train into calibration and validation sets.
# x_train_cal, x_train_val, y_train_cal, y_train_val = train_test_split(x_train, y_train, test_size=0.15, random_state=42)
## Then we use the calibration set statistics to standardize the calibration, validation and test sets
#x_train_cal_scaled, x_test_scaled = standardize_column(x_train_cal, x_test)
#_, x_train_val_scaled = standardize_column(x_train_cal, x_train_val)

x_train_scaled, x_test_scaled = standardize_column(x_train, x_test) 
# this will be slightly different than the previous one because we are using the whole train set to 
# standardize the test set. This is not a problem since we are not using the test set during training.


###### Callbacks to use during training
## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
progressbar = TqdmCallback()
#### Use this in the case of a validation partition (monitors the val_loss)
# 
# early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
## Note that the min_lr in rdlr is the one found in the LRFinder test
# rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
## Define the model name to save the best model weights.
# MODEL_NAME=f'study1_cnn_best_full.keras'
# checkpointer = ModelCheckpoint(filepath=path+MODEL_NAME, monitor='val_loss', verbose=0, save_best_only=True)

#### Use this in the case of no validation partition (monitors the loss)
MODEL_NAME = 'study1_cnn_best_full.keras'
checkpointer = ModelCheckpoint(filepath=path+MODEL_NAME, monitor='loss', verbose=0, save_best_only=True)
early_stop = keras.callbacks.EarlyStopping(monitor='loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='loss', verbose=0)


################## Define some of the training hyperparameters
## Number of samples in each batch
BATCH_SIZE=64
## Learning rate (this is a heuristic value, it should be tuned)
LR=0.01
print('Adam learning rate = {}'.format(LR))
## Number of epochs to train the model
EPOCHS=400


## Taken from the best trial - set by hand or...
# NUM_FC_LAYERS = 3
# NUM_FC_UNITS = [240,192,64]
# FILTER_SIZE = 10
# DROPOUT_RATE = [0.17,0.065] ## we can set a fixed dropout rate
# L2_REG = 0.012

## ...load the best hyperparameters directly from the best study
best_trial = study_1.best_trial
NUM_FC_LAYERS = best_trial.params['num_FC_layers']
NUM_FC_UNITS = [best_trial.params['num_FC_UNITS_'+str(i)] for i in range(0,NUM_FC_LAYERS)]
FILTER_SIZE = best_trial.params['filter_size']
DROPOUT_RATE = [best_trial.params[f"DROPOUT_{i}"] for i in range(NUM_FC_LAYERS - 1)]
L2_REG = best_trial.params['reg_beta']

## Create the model. It is important to create a new model instance each time to make sure the weights are reset
cnn_2 = create_model_2(INPUT_DIMS,NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, L2_REG)

## Compile the model, using the Adam optimizer, the loss function Mean Squared Error (MSE) because this is a regression problem
cnn_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])

## train the model
cnn_2.fit(x_train_scaled, y_train, batch_size = BATCH_SIZE, shuffle=False, epochs = EPOCHS,
          callbacks=[rdlr, early_stop, checkpointer, progressbar], verbose=0)

print(f'\n Training completed... \n Loading best model weights from {path+MODEL_NAME}...')
## Load the best model weights
cnn_2.load_weights(path+MODEL_NAME)

## Make model predictions
y_train_pred = cnn_2.predict(x_train_scaled)
# y_train_val_pred = cnn_2.predict(x_train_val_scaled)
y_test_pred = cnn_2.predict(x_test_scaled)

## Use prediction metrics to compute the RMSE and R2 scores
R2_train = r2_score(y_train, y_train_pred)
rmse_train = root_mean_squared_error(y_train, y_train_pred)
# R2_train_val = r2_score(y_train_val, y_train_val_pred)
# rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)
R2_test = r2_score(y_test, y_test_pred)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print('\n---------------------------- FULL TRAINING RESULTS ----------------------------')
print('\n\t ERROR METRICS: \t TRAIN   \t\t TEST')
print(f'\t R2: \t\t\t {R2_train:.3f}  \t\t {R2_test:.3f}')
print(f'\t RMSE: \t\t\t {rmse_train:.3f} \t\t\t {rmse_test:.3f}' )

## Clear clutter from previous session
keras.backend.clear_session()
print('\n Keras backend cleared...')


In [ ]:
plot_prediction2(y_train, y_test, y_train_pred, y_test_pred, 'Optimized CNN2 (test)', savefig=False, figname=None)

<div class="alert alert-block alert-warning">
<b>Suggestion:</b> If you want to assess the robustness of the model, you can set the shuffle parameter to True in the fit method and initialize it with different random seeds (it requires tweaking the create_model2() function). You can then run it 10 or 20 times in a for loop and compute the mean and standard deviation of the performance metrics.<br>
</div>

#### 5.4.2) Using a 5k-fold cross-validation strategy

In this example, we increase the complexity of the search by using:
- k-fold Cross-validation strategy and use the CV means to make decisions. **The down side is that we increase the number of computed models *k* times and this has a heavy computational cost.**
- We use a TPE (Tree Parzen Estimator) algorithm (type of bayesian search)
- We not only optimize the hyperparameters of the layers, but also the architecture (by implementing flags that create layers as needed, within bounds). So, in fact we are performing a joint HPO and NAS (Neural Architecture Search)
Step 1) Create an objective function for the "optimizer to optimize". In this case the objective function will be the cv-rmse
This cell defines the Optuna objective used for Bayesian optimisation: it standardises the data, samples CNN hyperparameters (dense layout, filter size, dropout, L2), performs 5-fold cross-validation with callbacks, and returns the average validation RMSE.

In [ ]:
## We might need this extra library for implementing prunning during the hyperparameter optimization.
from optuna.integration import TFKerasPruningCallback

## Set the path where the computed models are saved
## Beware of path notation differences between windows (\\) and linux (\) systems
path="/content/models/"

## Define the data to train the model (change it here for tecator, tomato or pear datasets):
x_train = tecator_x_train.values
y_train = tecator_y_train.values
x_test = tecator_x_test.values
y_test = tecator_y_test.values
INPUT_DIMS = np.shape(x_train)[1]


## Standardize the train and test sets. We will split the x_train_scaled into the k-fold cross-validation sets later on.
## Here we use the mean and std of the full x_train to standardize the x_test set. 
x_train_scaled, x_test_scaled = standardize_column(x_train, x_test)


## Define the objective function to monitor during optimization, set the hyperparameters ranges, etc.
def objective(trial):

    print('\n\n-------------- TRIAL NUMBER: ',trial.number,'------------------')

    ## Clear clutter from previous session
    keras.backend.clear_session()
    reproducible_comp()

    ## In the follow steps we define the search space for each of the hyperparameters. This is done by using
    ## distributions/intervals of integers or floats from a min, to a max with a certain step.
    ##
    ## Number of FC layers (1 to 3 layers)
    NUM_FC_LAYERS = trial.suggest_int("num_FC_layers",1,3, step=1)
    ## Number of units per layer
    NUM_FC_UNITS = [int(trial.suggest_int("num_FC_UNITS_"+str(i), 32, 256, step=16)) for i in range(0,NUM_FC_LAYERS)]
    ## Filter size  (K_WIDTH)
    FILTER_SIZE = int(trial.suggest_int("filter_size", 5, 45, step=5))
    ## Dropout rate (DROPOUT)
    DROPOUT_RATE = [trial.suggest_float("DROPOUT_"+str(i), 0., 0.5, step=0.005) for i in range(0,NUM_FC_LAYERS-1)]
    ## L2 regularization
    REG_BETA = trial.suggest_float("reg_beta", 0, 0.05, step=0.001)


    ## We can add batch size to the hyperparameters to be optimized. This hyperparameter is not used directly by the model()
    ## but is used in the training phase.
    ## BATCH_SIZE = int(trial.suggest_int("batch_size", 32, 256, step=32))
    ## In this case we keep if fixed as before for simplicity. The same goes for the learning rate LR.

    BATCH_SIZE=64
    ## Learning rate (this is a heuristic value, it should be tuned)
    LR=0.01 #*BATCH_SIZE/256.
    EPOCHS=300

    ## Instantiate the model for a new set of hyperparameters. 
    ## This way we can use the same model for different hyperparameters
    model = create_model_2(INPUT_DIMS, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, REG_BETA)
    print('\n\ncreate_model(',INPUT_DIMS, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, REG_BETA,')')
    # model.summary()
    print('\n')

    ## Compile the model, use Adam() as optimizer, and mean squared error (mse) as loss function

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])

    ## Empty lists to store the training and validation metrics
    scores_cal_list = []
    scores_val_list = []

    # create KFold object
    kf = KFold(n_splits = 5, shuffle=True, random_state=42) ## shuffled with a fixed random state for reproducibility

    ## Loop for training the model 3 times under different calibration/validation splits
    for i, (cal_index, val_index) in enumerate(kf.split(x_train)):
        ## Clear each fold training session
        keras.backend.clear_session()
        reproducible_comp()

        #### Callbacks to use during training
        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
        ## Note that the min_lr in rdlr is the one found in the LRFinder test
        rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=0)
        ## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
        progressbar = TqdmCallback()
               
        ## Define the cal and val sets for this iteration
        # x_cal_scaled, x_val_scaled = x_train_scaled[cal_index], x_train_scaled[val_index]
        x_cal_scaled, x_val_scaled = standardize_column(x_train[cal_index], x_train[val_index])
        y_cal, y_val = y_train[cal_index], y_train[val_index]


        print('\nRunning iteration Trial(Run) ',trial.number,'(', i+1,') --------')

        ## Reinitialize the model weights for this iteration
        model = create_model_2(INPUT_DIMS, NUM_FC_LAYERS, NUM_FC_UNITS, FILTER_SIZE, DROPOUT_RATE, REG_BETA)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR), loss="mse", metrics=["mse"])
        
        ## define the model/study name. In this case we will save a model with the same name of its trial
        MODEL_NAME='study2_cnn_trial='+str(trial.number)+'('+str(i)+').keras'
        checkpointer = ModelCheckpoint(filepath=path+MODEL_NAME, monitor='val_loss', verbose=0, save_best_only=True)

        ## Define the prunning callback to stop the training and skip to next trial. DOES NOT WORK WELL WITH K-FOLD!
        # pruning_callback = TFKerasPruningCallback(trial,monitor="val_mse",)

        ## Train the model on cal data and validate it on val data
        model.fit(x_cal_scaled, y_cal, shuffle=False, batch_size = BATCH_SIZE, epochs = EPOCHS,\
                  validation_data = (x_val_scaled, y_val),\
                  callbacks=[rdlr, early_stop, checkpointer, progressbar],\
                  verbose=0)

        ## Load the best model weights
        model.load_weights(path+MODEL_NAME)

        ## Compute auxiliary metrics
        scores_cal = model.evaluate(x_cal_scaled, y_cal, verbose=0)
        print('Calibration RMSE = {}'.format(np.sqrt(scores_cal[1])))
        scores_cal_list.append(np.sqrt(scores_cal[1]))

        scores_val = model.evaluate(x_val_scaled, y_val, verbose=0)
        fold_val_rmse = float(np.sqrt(scores_val[1]))
        print(f"Validation RMSE = {fold_val_rmse}")
        scores_val_list.append(fold_val_rmse)

        # Partial mean after the folds completed so far
        running_cv_rmse = float(np.mean(scores_val_list))

        # `i` is unique for each fold: 0, 1, 2, 3, 4
        trial.report(running_cv_rmse, step=i)

        if trial.should_prune():
             raise optuna.TrialPruned(
                f"Pruned after fold {i + 1}; "
                f"running CV RMSE = {running_cv_rmse:.4f}"
            )

        ## Clear clutter from previous session
        keras.backend.clear_session()
    ### END FOR LOOP

    ## Compute the mean of the metrics over the 5 trials
    scores_cal_mean = np.mean(scores_cal_list)
    scores_val_mean = np.mean(scores_val_list)
    scores_cal_std = np.std(scores_cal_list)
    scores_val_std = np.std(scores_val_list)

    print('\n\nMEAN OVER 5-fold CROSS-VALIDATION ----------------------------------')
    print('Calibration RMSE = {}+-{}'.format(scores_cal_mean, scores_cal_std))
    print('Validation RMSE = {}+-{}\n\n'.format(scores_val_mean, scores_val_std))

    ## The metric we want to minimize. In this case we are chosing weighted sum between tuning and cal metrics
    return scores_val_mean

In [ ]:
## Unique identifier of the study
STUDY_NAME = "study_2"
LOCAL_DB_DIR = "/content/optuna_dbs"
os.makedirs(LOCAL_DB_DIR, exist_ok=True)
LOCAL_DB_PATH = f"{LOCAL_DB_DIR}/{STUDY_NAME}.db"
## Name of the SQL data base where optuna saves the trials, values, etc...
## The optimization and logs the results into a database .db file that can be monitored in real time with the optuna-dashboard
STORAGE_NAME = "sqlite:///{}".format(LOCAL_DB_PATH)

## Create a study that will, in this case minimize the objective function previously defined
## We use TPE with a cte seed (for reproducibility), consider_endpoints = True (to consider the min and max values of the search space).
study_2 = optuna.create_study(study_name=STUDY_NAME, 
                              storage=STORAGE_NAME, 
                              direction='minimize', 
                              sampler = optuna.samplers.TPESampler(seed = 123,
                                                                   consider_endpoints = True,
                                                                   multivariate = True,
                                                                   n_startup_trials = 55,
                                                                   warn_independent_sampling=False),
                              load_if_exists=True)

## Add a hyperparameter set that we that works more or less well to the study
## In this case we are adding the best hyperparameters found in study_1 to the study_2. This is useful to start the optimization from a good point
study_2.enqueue_trial({'num_FC_layers': 3, 
                       'num_FC_UNITS_0': 144, 
                       'num_FC_UNITS_1': 192, 
                       'num_FC_UNITS_2': 208, 
                       'filter_size': 10, 
                       'DROPOUT_0': 0.005, 
                       'DROPOUT_1': 0.08, 
                       'reg_beta': 0.018000000000000002})
## NOTE: Adding this best trial as a starting point is not strictly necessary. It can point the optimizer into a good direction, but it can also bias the search space.


## Start optimization with a budget of n_trials
study_2.optimize(objective, n_trials = 2, show_progress_bar=True)

Check the best trial and its hyperparameters.

In [ ]:
## loat the precomputed study_2
DB_PATH_PRECOMPUTED = f"{data_path}/models/precomputed/study_2.db"
study_2 = optuna.load_study(study_name="study_2", storage=f"sqlite:///{DB_PATH_PRECOMPUTED}")

## load study (if it was not loaded before)
# study_2 = optuna.load_study(study_name="study_2", storage=f"sqlite:///{DB_PATH}")
print('Best trial:',study_2.best_trial.number)
print('Best trial value:',study_2.best_trial.value)
print('Best trial hyperparameters:', study_2.best_trial.params)

Predict using the best model from the HPO (done with 5-fold cross-validation). For trial 124 we have 5 different saved models, one for each fold.

In [ ]:
## Load the model from the best trial saved during the HPO
best_trial_number2 = study_2.best_trial.number

## Load each of the 5 models trained in the HPO best trial and compute the metrics for each of them.
for i in range(5):
    ## Load precomputed best model
    best_model2 = tf.keras.models.load_model(data_path+'/models/precomputed/study2_cnn_trial='+f'{best_trial_number2}({i}).keras')
    ## or your own
    # best_model2 = tf.keras.models.load_model(path+f'study2_cnn_trial={best_trial_number2}({i}).keras')


    ## Make model predictions
    y_train_cal_pred = best_model2.predict(x_train_cal_scaled)
    y_train_val_pred = best_model2.predict(x_train_val_scaled)
    y_test_pred = best_model2.predict(x_test_scaled)

    ## Use prediction metrics to compute the RMSE and R2 scores
    R2_train_cal = r2_score(y_train_cal, y_train_cal_pred)
    rmse_train_cal = root_mean_squared_error(y_train_cal, y_train_cal_pred)
    R2_train_val = r2_score(y_train_val, y_train_val_pred)
    rmse_train_val = root_mean_squared_error(y_train_val, y_train_val_pred)
    R2_test = r2_score(y_test, y_test_pred)
    rmse_test = root_mean_squared_error(y_test, y_test_pred)

    print('\n---------------------------- MODEL FROM BEST TRIAL: ',best_trial_number2,'(',i,') ----------------------------')

    print('\n\t ERROR METRICS: \t CALIB  \t\t VALID \t\t TEST')
    print(f'\t R2: \t\t\t {R2_train_cal:.3f}  \t\t {R2_train_val:.3f} \t\t {R2_test:.3f}')
    print(f'\t RMSE: \t\t\t {rmse_train_cal:.3f} \t\t\t {rmse_train_val:.3f} \t\t {rmse_test:.3f}' )

If we wanted to pick a model from these 5, we should choose the one with the lowest "VALID" RMSE -> 124 ( 4 ). From looking at the the performance metrics we can have an idea about the variability of the model across different folds. That is a data induced variability, which is different than running the same model, on the same data train/val split with different initialization seeds. This is a model induced variability. Both types of variability are important to understand the robustness of the model and its generalization capabilities.

## 6) CNN Explainability

### SHAP values

SHAP values are a powerful tool for interpreting machine learning models, including CNNs. They provide insights into how each feature (in this case, each wavelength in the NIR spectrum) contributes to the model's predictions. By calculating SHAP values, we can visualize which parts of the spectrum are most influential in determining the predicted concentration of the chemical compound. In this section we will compare the CNN derived SHAP values with the PLS VIP scores to see if the CNN is learning similar spectral features as the PLS model. For that we will use the same CNN model that we found on the HPO but, we will retrain it using the SNV preprocessed spectra (the same preprocessing used for the PLS model). This will allow us to compare the SHAP values with the PLS VIP scores on the same scale.

Lets compute the SHAP values for our model cnn_2. For the example we will use the pre-trained model from the HPO, that we will load and use to compute the SHAP values on the Tecator dataset.

In [ ]:
## Load the best precomputed model (structure and weights)
cnn2 = tf.keras.models.load_model(data_path+'/models/precomputed/study2_cnn_trial=124(4).keras')

These lines (kept commented) show how to compute RMSE and R² for calibration, validation, and test sets manually; uncomment them if you want to inspect the metrics outside the automated routines.


In [ ]:
## Define the data to use: the SNV preprocessed data. Note that this data was not used in the HPO study_2. 
## tecator_x_train_snv and tecator_x_test_snv are the SNV preprocessed data (see first cell of PLS section)
x_train = tecator_x_train_snv
y_train = tecator_y_train.values
x_test = tecator_x_test_snv
y_test = tecator_y_test.values
INPUT_DIMS = np.shape(x_train)[1]

## Standardize the train and test sets. 
## Here we use the mean and std of the full x_train to standardize the x_test set. 
x_train_scaled, x_test_scaled = standardize_column(x_train, x_test)

#### Callbacks to use during training
early_stop = keras.callbacks.EarlyStopping(monitor='loss', min_delta=1e-3, patience=50, mode='auto', restore_best_weights=True)
## Note that the min_lr in rdlr is the one found in the LRFinder test
rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='loss', verbose=0)
## This callback draws small progress bar in the screen for each training session. Its useful to check the progress of the task
progressbar = TqdmCallback()
## define the model/study name. In this case we will save a model with the same name of its trial
MODEL_NAME='cnn2_SNV.keras'
checkpointer = ModelCheckpoint(filepath=path+MODEL_NAME, monitor='loss', verbose=0, save_best_only=True)

## initialize the model from scratch (with the same structure as the best model found in study_2)
# tf.keras.backend.clear_session()
# reproducible_comp()
# cnn2 = create_model_2(INPUT_DIMS, 3, [144,224,256], 10, [0.315,0.035], 0.027)
# cnn2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005), loss="mse", metrics=["mse"])


## Note that (for simplification purposes) we are not instantiating a new model and we are starting the optimization from an already 
## trained model (on Column standardized data). This can be interpreted as a sort of transfer learning, where we are using the weights 
## of a model trained on one dataset (column standardized) to initialize the training of a model on another dataset (SNV preprocessed).
## In this case, we are not freezing any layers, so the model will be fully trained on the new dataset (SNV preprocessed).
cnn2.fit(x_train_scaled, y_train, batch_size = 64, shuffle=False, epochs = 350,
          callbacks=[rdlr, early_stop, checkpointer, progressbar], verbose=0)

cnn2.load_weights(path+MODEL_NAME)
## Make model predictions
y_train_pred = cnn2.predict(x_train_scaled)
y_test_pred = cnn2.predict(x_test_scaled)

## Use prediction metrics to compute the RMSE and R2 scores
R2_train = r2_score(y_train, y_train_pred)
rmse_train = root_mean_squared_error(y_train, y_train_pred)
R2_test = r2_score(y_test, y_test_pred)
rmse_test = root_mean_squared_error(y_test, y_test_pred)

print('\n---------------------------- MODEL METRICS TRAIN/TEST ----------------------------')

print('\n\t ERROR METRICS: \t TRAIN  \t\t TEST')
print(f'\t R2: \t\t\t {R2_train:.3f}  \t\t {R2_test:.3f}')
print(f'\t RMSE: \t\t\t {rmse_train:.3f} \t\t\t {rmse_test:.3f}' )

It seems that SNV actually improves the performance of the CNN model even though we did not optimize the preprocessing for the CNN. This is an interesting finding, as it suggests that the CNN is able to learn from the preprocessed spectra and extract relevant features for predicting the target variable.

We import SHAP, create a `GradientExplainer` for `cnn2`, and compute SHAP values on the test set using a subset of training samples as the background distribution. SHAP uses the train set (or a subset of it) to estimate the expected value of the model's output, which is then used to calculate the contribution of each feature to the predictions on the test set.

In [ ]:
import shap

## Learn the SHAP values for the CNN2 model. Note that we are using the GradientExplainer, which is suitable for deep learning models.
shap_cnn2 = shap.GradientExplainer(cnn2, x_train_scaled[:,:])


# Compute test-set SHAP values: return (samples, features, outputs). Normalize both to 2D.
shap_test_values_cnn2 = shap_cnn2.shap_values(x_test_scaled)

if shap_test_values_cnn2.ndim == 3 and shap_test_values_cnn2.shape[-1] == 1:
    shap_test_values_cnn2 = shap_test_values_cnn2[..., 0]


A quick shape check confirms the dimensionality of the CNN SHAP outputs before we start plotting them. It returns the SHAP values for each sample in the test set across all input features (wavelengths).


In [ ]:
print('CNN2 SHAP array:', shap_test_values_cnn2.shape)

Mean absolute SHAP values are plotted across wavelengths so you can identify the spectral regions that drive the CNN predictions; the mean spectrum is overlaid for context.


In [ ]:
## Plot CNN2 importance and the raw mean Tecator spectrum on separate y-axes.
## First we compute the mean absolute SHAP values for each feature (wavelength) across all test samples.
mean_abs_shap_cnn2 = np.mean(np.abs(shap_test_values_cnn2), axis=0)


fig, ax_shap = plt.subplots(figsize=(12, 4))
ax_shap.plot(tecator_w, mean_abs_shap_cnn2, 'r-', alpha=0.8, label='CNN2 mean |SHAP|')
ax_shap.set_title('SHAP feature importance for CNN2')
ax_shap.set_xlabel('Wavelength (nm)')
ax_shap.set_ylabel('Mean |SHAP value|', color='r')
ax_shap.tick_params(axis='y', labelcolor='r')
ax_shap.grid(axis='x', alpha=0.3)

ax_spectrum = ax_shap.twinx()
ax_spectrum.plot(tecator_w, np.mean(x_train, axis=0), 'g-', alpha=0.45, label='Mean spectrum')
ax_spectrum.set_ylabel('Mean absorbance', color='g')
ax_spectrum.tick_params(axis='y', labelcolor='g')

lines = ax_shap.get_lines() + ax_spectrum.get_lines()
ax_shap.legend(lines, [line.get_label() for line in lines], frameon=False)
fig.tight_layout()
plt.show()

Compute the SHAP and VIP values of PLS model. This is a slow process due to the use of Kernel Explainer (standard permutation kernel)
For comparison with the CNN, we fit a PLS model, compute its VIP scores, and use SHAP (with a shuffled background) to extract feature attributions on the derivative data.



In [ ]:
# best_LV, CV_RMSE = pls_optimization_cv_stop2(x_train_scaled, tecator_y_train, nmax=20, plot_opt=True, stop_criteria=0.001)
# print('Chosen number of LV:', best_LV, '\nCV RMSE:', CV_RMSE)

pls_model = PLSRegression(n_components=9, scale=True)
pls_model.fit(x_train, y_train)
# ## compute PLS vip scores
pls_vip = vip(pls_model)

_=pls_prediction_metrics2(tecator_w, x_train, y_train, x_test, y_test,
                            'Std','Moisture', lv=9,
                            plot_pred=True, plot_vip=True)


The SHAP profile obtained from the CNN model is plotted alongside the mean spectrum and PLS VIP scores, enabling a side-by-side interpretation.


In [ ]:
## Plot the PLS VIP scores and the mean spectra on separate y-axes.
plt.figure(figsize=(12,4))
ax1 = plt.gca()
ax1.plot(tecator_w, mean_abs_shap_cnn2,  'r-', alpha=0.7, label='SHAP values')
ax1.tick_params(axis='y', labelcolor='r')  
ax1.set_ylabel('Mean abs(SHAP values)', color='r')
# plt.plot(tecator_w,3*np.mean(tecator_x_train_snv, axis=0), 'g-', lw=6,alpha=0.5, label='mean spectra')
plt.legend(loc=4)
## Add second axis with vip scores
ax2 = plt.gca().twinx()
ax2.plot(tecator_w, pls_vip, 'b-', lw=3, alpha=0.7, label='PLS VIP scores')
ax2.set_ylabel('PLS VIP scores', color='b')
ax2.tick_params(axis='y', labelcolor='b')   
ax2.hlines(y=1, xmin=tecator_w[0], xmax=tecator_w[-1], colors='k', linestyles='dashed', lw=2, alpha=0.5)
ax2.legend(loc='upper left')
plt.grid(axis='x')
plt.tight_layout()
plt.show()

**Some thoughts!**

For the PLS model, VIP > 1 occurs approximately in these broad regions: 850–892 nm, 916–938 nm and 964–992 nm.
The CNN has its strongest isolated SHAP peaks around: 902–914 nm, 932–942 nm, 980–986 nm, approximately 1004 nm and 1040–1048 nm.

The main agreement is therefore around 928–934 nm, with weaker regional agreement around 970–990 nm.

Chemically, this is plausible! Around 930 nm is associated strongly with fat absorption in meat. The broad 970–980 nm region is associated with water absorption and features near 1040 nm have also been associated with methyl/methylene groups in fatty acids. Because the target is moisture, the 970–980 nm agreement has an obvious direct interpretation. Importance around 930 or 1040 nm could mean that the model also uses fat-related information as an indirect predictor of moisture—plausible because fat and moisture vary together compositionally in meat. That latter interpretation is an inference, not proof of a causal moisture band.

**Why the CNN and PLS can differ**

The CNN has substantially better test performance in this case:<br>

CNN: $R^2=0.984, RMSE = 1.265$<br>
PLS: $R^2=0.965, RMSE = 1.949$<br>

This suggests the CNN may be using relationships unavailable to the linear PLS model, including nonlinear combinations or interactions between wavelength regions. However, the current CNN was initialized from a model trained on differently processed data and then fine-tuned on SNV data. For a strong scientific conclusion, its SHAP profile should be checked across several from-scratch initializations or folds.
Also, adjacent NIR wavelengths are highly correlated. One model may distribute importance broadly across a band, while another assigns most of it to a few representative wavelengths. This helps explain the smooth VIP curve versus the jagged SHAP profile.




In [ ]:
###################### EXTRA CELL:  Compare CNN and PLS SHAP ##############################
### Compare CNN and PLS SHAP values using the same explicit inputs, background,         ###
### evaluation samples, and target units. The 10 nm panel aggregates raw SHAP           ###
### importance within predefined bands; it does not smooth the wavelength values.       ###
###########################################################################################


# import warnings
# import shap
# from scipy import stats
# from sklearn.cross_decomposition import PLSRegression

# shap_background = np.asarray(x_train_scaled, dtype=np.float32)
# shap_evaluation = np.asarray(x_test_scaled, dtype=np.float32)
# wavelengths = np.asarray(tecator_w, dtype=float)

# # CNN expected-gradient SHAP values. local_smoothing=0 keeps the attributions
# # unsmoothed; the fixed seed makes the Monte Carlo estimate reproducible.
# cnn_explainer = shap.GradientExplainer(
#     cnn2, shap_background, local_smoothing=0
# )
# with warnings.catch_warnings():
#     warnings.filterwarnings(
#         'ignore',
#         message='The structure of `inputs` doesn.*',
#         category=UserWarning,
#     )
#     cnn_shap_values = cnn_explainer.shap_values(
#         shap_evaluation, nsamples=1000, rseed=42
#     )
# if isinstance(cnn_shap_values, list):
#     cnn_shap_values = cnn_shap_values[0]
# cnn_shap_values = np.asarray(cnn_shap_values)
# if cnn_shap_values.ndim == 3 and cnn_shap_values.shape[-1] == 1:
#     cnn_shap_values = cnn_shap_values[..., 0]

# # Fit the equivalent nine-component PLS model directly on the explicitly
# # standardized SNV spectra. scale=False avoids scaling the same inputs twice.
# pls_shap_model = PLSRegression(n_components=9, scale=False)
# pls_shap_model.fit(shap_background, y_train)
# pls_masker = shap.maskers.Independent(
#     shap_background, max_samples=shap_background.shape[0]
# )
# pls_explainer = shap.LinearExplainer(pls_shap_model, pls_masker)
# pls_shap_values = np.asarray(pls_explainer.shap_values(shap_evaluation))
# if pls_shap_values.ndim == 3 and pls_shap_values.shape[-1] == 1:
#     pls_shap_values = pls_shap_values[..., 0]

# expected_shape = (shap_evaluation.shape[0], shap_evaluation.shape[1])
# assert cnn_shap_values.shape == expected_shape
# assert pls_shap_values.shape == expected_shape

# # Global wavelength importance: average absolute contribution over the same
# # test spectra. Both profiles remain in moisture-prediction units.
# cnn_mean_abs_shap = np.mean(np.abs(cnn_shap_values), axis=0)
# pls_mean_abs_shap = np.mean(np.abs(pls_shap_values), axis=0)
# wavelength_rho = stats.spearmanr(
#     cnn_mean_abs_shap, pls_mean_abs_shap
# ).statistic

# # Predefine contiguous 10 nm bands beginning at the nearest lower multiple
# # of 10 nm, then sum the unsmoothed wavelength importances in each band.
# band_width_nm = 10.0
# band_start = band_width_nm * np.floor(wavelengths.min() / band_width_nm)
# band_stop = band_width_nm * np.ceil(wavelengths.max() / band_width_nm)
# band_edges = np.arange(band_start, band_stop + band_width_nm, band_width_nm)
# band_centres = 0.5 * (band_edges[:-1] + band_edges[1:])
# band_index = np.digitize(wavelengths, band_edges[1:-1])
# number_of_bands = len(band_centres)
# cnn_band_shap = np.bincount(
#     band_index, weights=cnn_mean_abs_shap, minlength=number_of_bands
# )
# pls_band_shap = np.bincount(
#     band_index, weights=pls_mean_abs_shap, minlength=number_of_bands
# )
# band_rho = stats.spearmanr(cnn_band_shap, pls_band_shap).statistic

# # Display the wavelength-resolution and band-level comparisons side by side.
# fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))

# axes[0].plot(
#     wavelengths, cnn_mean_abs_shap, color='tab:red', lw=1.6, label='CNN SHAP'
# )
# axes[0].plot(
#     wavelengths, pls_mean_abs_shap, color='tab:blue', lw=2.0, label='PLS SHAP'
# )
# axes[0].set(
#     xlabel='Wavelength (nm)',
#     ylabel='Mean |SHAP value| (moisture units)',
#     title=f'All wavelengths (Spearman ρ = {wavelength_rho:.2f})',
# )
# axes[0].grid(alpha=0.25)
# axes[0].legend(frameon=False)

# bar_width = 0.38 * band_width_nm
# axes[1].bar(
#     band_centres - bar_width / 2,
#     cnn_band_shap,
#     width=bar_width,
#     color='tab:red',
#     alpha=0.75,
#     label='CNN SHAP',
# )
# axes[1].bar(
#     band_centres + bar_width / 2,
#     pls_band_shap,
#     width=bar_width,
#     color='tab:blue',
#     alpha=0.75,
#     label='PLS SHAP',
# )
# axes[1].set(
#     xlabel='Predefined 10 nm wavelength band (nm)',
#     ylabel='Sum of mean |SHAP value| within band',
#     title=f'10 nm bands (Spearman ρ = {band_rho:.2f})',
# )
# axes[1].set_xticks(band_centres)
# axes[1].set_xticklabels(
#     [f'{int(left)}–{int(right)}' for left, right in zip(band_edges[:-1], band_edges[1:])],
#     rotation=60,
#     ha='right',
# )
# axes[1].grid(axis='y', alpha=0.25)
# axes[1].legend(frameon=False)

# fig.suptitle(
#     'CNN versus PLS SHAP importance on the same standardized SNV spectra',
#     y=1.02,
# )
# fig.tight_layout()
# plt.show()

# print(f'Wavelength-level Spearman correlation: {wavelength_rho:.3f}')
# print(f'10 nm band-level Spearman correlation: {band_rho:.3f}')


## 7) Classification and unsupervised representation learning

In this final section we reuse the Tuttifrutti NIR spectra for two different machine-learning tasks:

1. **Supervised classification:** predict the fruit type from a spectrum using a 1D CNN.
2. **Unsupervised representation learning:** train a convolutional autoencoder to reconstruct spectra, then inspect its internal features with PCA.


### 7.1) Load and inspect the Tuttifrutti dataset

In this dataset each row is one the already preprocessed second-derivative NIR spectrum. The first 105 columns are spectral intensities indexed by wavelength; the final two columns are:

- `DM`: dry-matter content, which would be the target in a regression problem but is not used here.
- `Fruit`: the class label used by the classifier.

The supplied files contain Apple (0), Kiwi (1), Mango (2), and Pear (3).

In [ ]:
# Load the predefined train and test files into Pandas DataFrames.
# Keeping the official split lets us compare models on the same unseen samples.
ttfruit_train = pd.read_csv(f"{data_path}/tuttifruti/tuttifruti_train.csv")
ttfruit_test = pd.read_csv(f"{data_path}/tuttifruti/tuttifruti_test.csv")

# The first n-2 columns are model inputs. The two metadata/target columns at the end
# are kept separately so neither DM nor the fruit label leaks into the spectra.
ttfruit_x_train = ttfruit_train.iloc[:, 0:-2]
ttfruit_y_train = ttfruit_train.iloc[:, -2]          # DM regression target (unused here)
ttfruit_class_train = ttfruit_train.iloc[:, -1]      # Fruit classification target

ttfruit_x_test = ttfruit_test.iloc[:, 0:-2]
ttfruit_y_test = ttfruit_test.iloc[:, -2]
ttfruit_class_test = ttfruit_test.iloc[:, -1]

# CSV headers are strings. Convert the spectral headers to floating-point
# wavelengths so they can be used as the physical x-axis in later plots.
tuttifruti_w = np.array([float(column) for column in ttfruit_x_train.columns])

# Human-readable names make plots and reports easier to interpret than integers.
fruit_name_lookup = {0: 'Apple', 1: 'Kiwi', 2: 'Mango', 3: 'Pear'}

# These checks make the dimensions and available labels explicit to students.
print('Training spectra:', ttfruit_x_train.shape)
print('Test spectra:', ttfruit_x_test.shape)
print('Training class counts:')
print(ttfruit_class_train.value_counts().sort_index())

#### Visual sanity check

Before fitting a model, lets plot the spectra grouped by fruit class. This checks that all samples share the same wavelength grid and gives a first impression of within-class variation and between-class spectral structure. 

This plot is exploratory only: the fruit labels are used to choose line colors, not to transform the spectra.

In [ ]:
# Plot the test spectra by fruit type. Using the test labels here is only for
# visualization; no information from this plot is passed into either model.
plt.figure(figsize=(12, 4))

fruit_colors = {
    'apple': 'red',
    'kiwi': 'green',
    'mango': 'orange',
    'pear': 'blue',
}

# Plot one class at a time so all spectra from that class share a color.
for fruit_class in sorted(ttfruit_class_test.unique()):
    # Boolean masks select only rows belonging to the current fruit class.
    class_mask = ttfruit_class_test == fruit_class
    class_name = fruit_name_lookup.get(int(fruit_class), str(fruit_class))
    color = fruit_colors[class_name.lower()]

    # Pandas stores samples in rows. Transposing makes each sample a line whose
    # x-axis is wavelength. Alpha blending reveals regions with many overlaps.
    plt.plot(
        tuttifruti_w,
        ttfruit_x_test[class_mask].T,
        color=color,
        alpha=0.35,
    )

    # Add one empty line per class to create a clean legend entry instead of
    # hundreds of repeated labels—one for every spectrum.
    plt.plot([], [], color=color, label=class_name)

plt.xlabel('Wavelength (nm)')
plt.ylabel('Second-derivative intensity')
plt.title('Tuttifrutti test spectra colored by fruit type')
plt.legend(title='Fruit class', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 7.2) Supervised classification with a simple CNN

The classifier follows the same structure as the earlier regression CNN:

`spectrum → reshape → 1D convolution → flatten → dense layer → output`

The final layer is the key difference. Regression used one linear output, whereas multiclass classification uses one output per fruit and a `softmax` activation. Softmax converts the output scores into probabilities that sum to one. Because the target labels are stored as integers, the matching loss is sparse categorical cross-entropy.

The original training data are split into calibration and validation subsets. Calibration samples update the model weights; validation samples control early stopping and learning-rate reduction; the independent test set is used only for the final evaluation.

In [ ]:
# Prepare calibration/validation/test arrays and define the CNN classifier.
import warnings
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Reset all random generators defined earlier in the tutorial. This makes the
# split and initial model weights repeatable across runs on the same software stack.
reproducible_comp()

# The Tuttifruti files use the four class indices expected by sparse
# categorical cross-entropy: 0, 1, 2, and 3.
ttfruit_class_labels = np.array([0, 1, 2, 3], dtype=np.int32)
ttfruit_class_names = [
    fruit_name_lookup.get(int(class_label), str(class_label))
    for class_label in ttfruit_class_labels
]

# Warn if either data file contains a label outside the four expected classes.
expected_class_labels = set(ttfruit_class_labels)
observed_class_labels = set(ttfruit_class_train.unique()) | set(ttfruit_class_test.unique())
unexpected_class_labels = observed_class_labels - expected_class_labels
if unexpected_class_labels:
    warnings.warn(
        f'Unexpected class labels: {sorted(unexpected_class_labels)}. '
        f'Expected only: {sorted(expected_class_labels)}.',
        UserWarning,
    )

# The file labels already match the indices used by Keras, so no encoding is needed.
ttfruit_y_train_encoded = ttfruit_class_train.to_numpy(dtype=np.int32)
ttfruit_y_test_encoded = ttfruit_class_test.to_numpy(dtype=np.int32)



# Split before fitting the scaler to prevent validation information from affecting
# preprocessing. Stratification preserves the fruit proportions in both subsets.
ttfruit_x_cal, ttfruit_x_val, ttfruit_y_cal, ttfruit_y_val = train_test_split(
    ttfruit_x_train,
    ttfruit_y_train_encoded,
    test_size=0.20,
    random_state=42,
    stratify=ttfruit_y_train_encoded,
)

# Fit one mean and standard deviation per wavelength using calibration samples only.
# The exact same transformation is then applied to validation and test spectra.
ttfruit_scaler = StandardScaler().fit(ttfruit_x_cal)
ttfruit_x_train_scaled = ttfruit_scaler.transform(ttfruit_x_train).astype(np.float32)
ttfruit_x_test_scaled = ttfruit_scaler.transform(ttfruit_x_test).astype(np.float32)
ttfruit_x_cal_scaled = ttfruit_scaler.transform(ttfruit_x_cal).astype(np.float32)
ttfruit_x_val_scaled = ttfruit_scaler.transform(ttfruit_x_val).astype(np.float32)


def create_tuttifruti_classifier(
    input_dims, num_classes, filter_size=15, dense_units=64, reg_beta=1e-4
):
    """Create the earlier tutorial CNN with a multiclass softmax output."""
    # L2 regularization discourages excessively large weights. He initialization
    # is well suited to layers followed by ReLU-like activations.
    kernel_regularizer = tf.keras.regularizers.L2(reg_beta)
    kernel_initializer = tf.keras.initializers.HeNormal(seed=123)

    # Keras receives each spectrum as a flat vector (wavelengths,). Conv1D expects
    # (positions, channels), so add a single spectral channel with Reshape.
    inputs = layers.Input(shape=(input_dims,), name='INPUT')
    x = layers.Reshape((input_dims, 1), name='RESHAPE')(inputs)

    # The 1D kernel slides along wavelength and learns a local spectral pattern.
    # 'same' padding preserves the original number of wavelength positions.
    x = layers.Conv1D(
        filters=1,
        kernel_size=filter_size,
        strides=1,
        padding='same',
        kernel_initializer=kernel_initializer,
        kernel_regularizer=kernel_regularizer,
        bias_regularizer=kernel_regularizer,
        activation=layers.LeakyReLU(negative_slope=0.2),
        name='CONVOLUTIONAL',
    )(x)

    # Flatten converts the convolutional response into a vector. The dense layer
    # combines information from different wavelength regions before classification.
    x = layers.Flatten(name='FLATTEN')(x)
    x = layers.Dense(
        dense_units,
        kernel_initializer=kernel_initializer,
        kernel_regularizer=kernel_regularizer,
        bias_regularizer=kernel_regularizer,
        activation=layers.LeakyReLU(negative_slope=0.2),
        name='DENSE',
    )(x)

    # One softmax unit per class produces P(class | spectrum) values summing to one.
    class_output = layers.Dense(
        num_classes,
        activation='softmax',
        name='CLASS_OUTPUT',
    )(x)
    return Model(inputs=inputs, outputs=class_output, name='TUTTIFRUTI_CLASSIFIER')


# Clear previously constructed TensorFlow graphs before creating a fresh model.
keras.backend.clear_session()
reproducible_comp()
ttfruit_classifier = create_tuttifruti_classifier(
    input_dims=ttfruit_x_cal_scaled.shape[1],
    num_classes=len(ttfruit_class_labels),
)

# Adam updates the weights, cross-entropy measures probability error, and accuracy
# reports the fraction of samples assigned to the correct class.
ttfruit_classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

print('Class mapping:', dict(zip(ttfruit_class_labels, ttfruit_class_names)))
print(
    'Calibration / validation / test:',
    ttfruit_x_cal_scaled.shape,
    ttfruit_x_val_scaled.shape,
    ttfruit_x_test_scaled.shape,
)
ttfruit_classifier.summary()

#### Train the classifier and predict the test set

Training is allowed to run for at most 200 epochs, but two callbacks usually stop or slow it earlier:

- `EarlyStopping` restores the weights from the epoch with the best validation loss.
- `ReduceLROnPlateau` lowers the learning rate when validation loss stops improving.

After training, the learning curves are plotted to compare calibration and validation behavior. The model then returns one probability vector per test spectrum. `argmax` selects the class with the largest probability, while the largest probability itself is reported as the model confidence.

In [ ]:
keras.backend.clear_session()
reproducible_comp()

# Train the classifier, predict the test classes, and inspect performance.

# Both callbacks monitor validation loss, which is not used for gradient updates.
# Restoring the best weights avoids keeping a later, overfitted epoch.
ttfruit_classifier_callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=0,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        patience=8,
        factor=0.5,
        min_lr=1e-6,
        verbose=0,
    ),
]

# Only calibration samples update the weights. Validation samples provide an
# independent learning signal for the callbacks after each epoch.
ttfruit_classifier_history = ttfruit_classifier.fit(
    ttfruit_x_cal_scaled,
    ttfruit_y_cal,
    validation_data=(ttfruit_x_val_scaled, ttfruit_y_val),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=ttfruit_classifier_callbacks,
    verbose=0,
)

# Plot loss and accuracy together. A widening calibration-validation gap can be
# evidence of overfitting even if calibration performance continues to improve.
history_epochs = np.arange(1, len(ttfruit_classifier_history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history_epochs, ttfruit_classifier_history.history['loss'], label='calibration')
axes[0].plot(history_epochs, ttfruit_classifier_history.history['val_loss'], label='validation')
axes[0].set(xlabel='Epoch', ylabel='Cross-entropy', title='Classifier loss')
axes[0].legend(frameon=False)
axes[1].plot(history_epochs, ttfruit_classifier_history.history['accuracy'], label='calibration')
axes[1].plot(history_epochs, ttfruit_classifier_history.history['val_accuracy'], label='validation')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Classifier accuracy')
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

# predict() returns a probability for every class. argmax converts each probability
# vector into an internal class index; the lookup converts it back to a fruit name.
ttfruit_test_probabilities = ttfruit_classifier.predict(ttfruit_x_test_scaled, verbose=0)
ttfruit_test_predicted_index = np.argmax(ttfruit_test_probabilities, axis=1)
ttfruit_test_predicted_label = ttfruit_class_labels[ttfruit_test_predicted_index]
ttfruit_test_predicted_name = np.array(ttfruit_class_names)[ttfruit_test_predicted_index]
ttfruit_test_true_name = np.array(ttfruit_class_names)[ttfruit_y_test_encoded]
ttfruit_test_confidence = np.max(ttfruit_test_probabilities, axis=1)

# Evaluate the untouched test set once, after model development is complete.
ttfruit_test_loss, ttfruit_test_accuracy = ttfruit_classifier.evaluate(
    ttfruit_x_test_scaled, ttfruit_y_test_encoded, verbose=0
)
print(f'Test loss: {ttfruit_test_loss:.4f}')
print(f'Test accuracy: {ttfruit_test_accuracy:.3%}')

# Precision, recall, and F1 are reported per fruit, which is important when class
# counts are unequal and overall accuracy could hide a weak minority class.
print('\nClassification report:')
print(
    classification_report(
        ttfruit_y_test_encoded,
        ttfruit_test_predicted_index,
        labels=np.arange(len(ttfruit_class_names)),
        target_names=ttfruit_class_names,
        digits=3,
        zero_division=0,
    )
)

# Show a bounded sample of individual predictions rather than printing all 600 rows.
ttfruit_predictions = pd.DataFrame(
    {
        'true_class': ttfruit_test_true_name,
        'predicted_class': ttfruit_test_predicted_name,
        'confidence': ttfruit_test_confidence,
    }
)
display(ttfruit_predictions.head(20))

# In the confusion matrix, rows are true fruits and columns are predicted fruits.
# Off-diagonal entries therefore reveal which pairs of fruits the model confuses.
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(
    ttfruit_y_test_encoded,
    ttfruit_test_predicted_index,
    labels=np.arange(len(ttfruit_class_names)),
    display_labels=ttfruit_class_names,
    cmap='Blues',
    colorbar=False,
    ax=ax,
)
ax.set_title('Tuttifrutti test-set confusion matrix')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

#### How to read the classification results

- **Learning curves:** calibration and validation curves should improve together. A calibration curve that keeps improving while validation degrades suggests overfitting.
- **Precision:** among samples predicted as a fruit, the fraction that truly belongs to that fruit.
- **Recall:** among all true samples of a fruit, the fraction correctly found by the model.
- **F1-score:** a balance between precision and recall.
- **Confusion matrix:** a strong classifier concentrates counts on the diagonal.

The reported confidence is simply the largest softmax probability. It is useful for ranking uncertain predictions, but it should not automatically be interpreted as a calibrated probability without a separate calibration analysis.

### 7.3) Unsupervised learning with a convolutional autoencoder

An autoencoder is trained to reproduce its input: the spectrum is both the input and the target. It contains two connected parts:

`spectrum → encoder → bottleneck feature map → decoder → reconstructed spectrum`

The encoder learns a lower-resolution representation that retains information needed for reconstruction. The decoder expands that representation back to 105 wavelengths. No fruit labels are passed to `fit`, so this is unsupervised representation learning. Labels will be introduced only after training to color the PCA visualization.

Here the bottleneck has shape `27 × 4`: 27 reduced spectral positions and four learned feature channels. It is a compact spatial representation, although its 108 scalar activations are not strictly fewer than the 105 input values.

In [ ]:
# Define the convolutional autoencoder and a separate encoder view.
def create_tuttifruti_autoencoder(input_dims):
    # Conv1D expects (wavelength positions, channels). Each spectrum has one channel.
    inputs = layers.Input(shape=(input_dims, 1), name='SPECTRUM_INPUT')

    # ------------------------------- Encoder -------------------------------
    # Convolutions learn local patterns; max-pooling halves spectral resolution.
    # 'same' padding rounds odd dimensions upward, giving 105 -> 53 -> 27 positions.
    x = layers.Conv1D(
        16, 7, padding='same', activation='relu', name='ENCODER_CONV_1'
    )(inputs)
    x = layers.MaxPooling1D(2, padding='same', name='ENCODER_POOL_1')(x)
    x = layers.Conv1D(
        8, 5, padding='same', activation='relu', name='ENCODER_CONV_2'
    )(x)
    x = layers.MaxPooling1D(2, padding='same', name='ENCODER_POOL_2')(x)

    # This named layer is the latent representation we will later extract and analyze.
    bottleneck = layers.Conv1D(
        4, 3, padding='same', activation='relu', name='BOTTLENECK'
    )(x)

    # ------------------------------- Decoder -------------------------------
    # Upsampling reverses the resolution reduction: 27 -> 54 -> 108 positions.
    x = layers.Conv1D(
        8, 3, padding='same', activation='relu', name='DECODER_CONV_1'
    )(bottleneck)
    x = layers.UpSampling1D(2, name='DECODER_UPSAMPLE_1')(x)
    x = layers.Conv1D(
        16, 5, padding='same', activation='relu', name='DECODER_CONV_2'
    )(x)
    x = layers.UpSampling1D(2, name='DECODER_UPSAMPLE_2')(x)

    # Standardized spectra contain positive and negative values, so the final
    # reconstruction uses a linear activation rather than ReLU or sigmoid.
    reconstruction = layers.Conv1D(
        1, 7, padding='same', activation='linear', name='RECONSTRUCTION_RAW'
    )(x)

    # Because 105 is not divisible by four, pooling and upsampling produce 108
    # positions. Crop only the extra positions to recover the exact input length.
    decoded_length = 4 * int(np.ceil(np.ceil(input_dims / 2) / 2))
    crop_right = decoded_length - input_dims
    if crop_right > 0:
        reconstruction = layers.Cropping1D(
            cropping=(0, crop_right), name='RECONSTRUCTION'
        )(reconstruction)

    # Both models share the same encoder weights. The full model reconstructs spectra;
    # the encoder model stops at BOTTLENECK and exposes its learned features.
    autoencoder = Model(inputs, reconstruction, name='TUTTIFRUTI_AUTOENCODER')
    encoder = Model(inputs, bottleneck, name='TUTTIFRUTI_ENCODER')
    return autoencoder, encoder


keras.backend.clear_session()
reproducible_comp()
ttfruit_autoencoder, ttfruit_encoder = create_tuttifruti_autoencoder(
    ttfruit_x_cal_scaled.shape[1]
)

# Mean squared error measures the average reconstruction difference at every
# wavelength. No classification metric is needed because no labels are predicted.
ttfruit_autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
)
ttfruit_autoencoder.summary()
print('Bottleneck feature-map shape:', ttfruit_encoder.output_shape)

#### Train the reconstruction objective

The scaled 2D matrices have shape `(samples, wavelengths)`. A singleton channel dimension is added to obtain `(samples, wavelengths, 1)` for `Conv1D`.

During training, the same calibration spectra are supplied as both `x` and `y`. Validation reconstruction loss controls the callbacks, while the test set remains untouched. After training, test reconstruction MSE measures generalization to unseen spectra, and an original/reconstructed pair provides a wavelength-by-wavelength visual check.

A low MSE is useful, but it does not prove that the bottleneck is chemically meaningful. The PCA analysis in the next step helps us inspect what structure the representation retained.

In [ ]:
# Train the autoencoder and check reconstruction quality on unseen spectra.

# Add the single channel required by Conv1D without changing spectral values.
ttfruit_x_cal_3d = ttfruit_x_cal_scaled[..., np.newaxis]
ttfruit_x_val_3d = ttfruit_x_val_scaled[..., np.newaxis]
ttfruit_x_train_3d = ttfruit_x_train_scaled[..., np.newaxis]
ttfruit_x_test_3d = ttfruit_x_test_scaled[..., np.newaxis]

# Use the same validation-based safeguards as the classifier, with a smaller
# min_delta because reconstruction MSE is on a different numerical scale.
ttfruit_autoencoder_callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, min_delta=1e-5, restore_best_weights=True, verbose=0,),
    ReduceLROnPlateau(monitor='val_loss', patience=8, factor=0.5, min_lr=1e-6, verbose=0,    ),
]
# Notice that x and y are identical: the model learns reconstruction, not class labels.
ttfruit_autoencoder_history = ttfruit_autoencoder.fit(ttfruit_x_cal_3d, ttfruit_x_cal_3d,
    validation_data=(ttfruit_x_val_3d, ttfruit_x_val_3d), epochs=200, batch_size=64,
    shuffle=True, callbacks=ttfruit_autoencoder_callbacks, verbose=0,)

# The test data are used only after training to estimate out-of-sample reconstruction.
ttfruit_test_reconstruction = ttfruit_autoencoder.predict(ttfruit_x_test_3d, verbose=0)
ttfruit_test_reconstruction_mse = np.mean((ttfruit_x_test_3d - ttfruit_test_reconstruction) ** 2)
print(f'Test reconstruction MSE (standardized spectra): {ttfruit_test_reconstruction_mse:.5f}')


# Compare optimization behavior and one representative reconstruction.
ae_epochs = np.arange(1, len(ttfruit_autoencoder_history.history['loss']) + 1)
example_index = 0
example_class_index = ttfruit_y_test_encoded[example_index]
example_class_name = ttfruit_class_names[example_class_index]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(ae_epochs, ttfruit_autoencoder_history.history['loss'], label='calibration')
axes[0].plot(ae_epochs, ttfruit_autoencoder_history.history['val_loss'], label='validation')
axes[0].set(xlabel='Epoch', ylabel='MSE', title='Autoencoder reconstruction loss')
axes[0].legend(frameon=False)

# Overlaying the curves shows where the decoder reproduces or smooths spectral detail.
axes[1].plot(tuttifruti_w, ttfruit_x_test_scaled[example_index], color='black', lw=1.5, label='original',)
axes[1].plot(tuttifruti_w, ttfruit_test_reconstruction[example_index, :, 0], color='tab:red', lw=1.5, alpha=0.8, label='reconstruction',)
axes[1].set(xlabel='Wavelength (nm)', ylabel='Standardized intensity', title=f'Test reconstruction: {example_class_name}',)
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

### 7.4) Explore bottleneck features with three-component PCA

The encoder returns a `27 × 4` feature map for every spectrum. PCA expects one feature vector per sample, so each map is flattened to 108 features. Those features are standardized because different bottleneck channels can have different numerical scales.

Both feature scaling and PCA are fitted only on the training bottlenecks. The already-fitted transformations are then applied to test bottlenecks, placing unseen samples in the same coordinate system without test leakage.

PCA finds orthogonal directions of decreasing variance. It does **not** use fruit labels and does not explicitly optimize class separation. The labels are added afterward solely to color and shape the points in the interactive Plotly visualization.

In [ ]:
# Extract bottleneck features and visualize three PCA components interactively.
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Run the shared encoder on every spectrum. Each output has shape (27, 4).
ttfruit_train_bottleneck = ttfruit_encoder.predict(ttfruit_x_train_3d, verbose=0)
ttfruit_test_bottleneck = ttfruit_encoder.predict(ttfruit_x_test_3d, verbose=0)

# Flatten each feature map while keeping samples as rows: (samples, 27 * 4).
ttfruit_train_features = ttfruit_train_bottleneck.reshape(len(ttfruit_x_train_3d), -1)
ttfruit_test_features = ttfruit_test_bottleneck.reshape(len(ttfruit_x_test_3d), -1)

# Fit preprocessing only on training features, then reuse it for test features.
# Feature standardization prevents high-amplitude bottleneck units from dominating PCA.
ttfruit_feature_scaler = StandardScaler().fit(ttfruit_train_features)
ttfruit_train_features_scaled = ttfruit_feature_scaler.transform(ttfruit_train_features)
ttfruit_test_features_scaled = ttfruit_feature_scaler.transform(ttfruit_test_features)

# Fit three components so every spectrum receives PC1, PC2, and PC3 coordinates.
ttfruit_bottleneck_pca = PCA(n_components=3, random_state=42)
ttfruit_train_pca = ttfruit_bottleneck_pca.fit_transform(ttfruit_train_features_scaled)
ttfruit_test_pca = ttfruit_bottleneck_pca.transform(ttfruit_test_features_scaled)

# explained_variance_ratio_ reports the fraction of training-feature variance
# represented by each displayed axis.
explained = 100 * ttfruit_bottleneck_pca.explained_variance_ratio_

# Keep colors and marker shapes consistent between training and test panels.
# Shape supplements color so class identity does not depend on color perception alone.
class_palette = ['#2F6B9A', '#8A9A3A', '#D9902F', '#C65C8A', '#6D7F2B']
class_symbols = ['circle', 'diamond', 'square', 'cross', 'x']
class_colors = {
    class_name: class_palette[index % len(class_palette)]
    for index, class_name in enumerate(ttfruit_class_names)
}

# Plot training and test scores side by side. Both scenes use the PCA coordinate
# system fitted on training data, so their cluster locations are directly comparable.
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=('Training set', 'Test set'),
    horizontal_spacing=0.04,
)

# Add one trace per class and split. Separate traces make legend filtering possible.
for column, (scores, encoded_labels, split_name) in enumerate(
    [
        (ttfruit_train_pca, ttfruit_y_train_encoded, 'Training'),
        (ttfruit_test_pca, ttfruit_y_test_encoded, 'Test'),
    ],
    start=1,
):
    for class_index, class_name in enumerate(ttfruit_class_names):
        class_mask = encoded_labels == class_index
        sample_indices = np.flatnonzero(class_mask)

        fig.add_trace(
            go.Scatter3d(
                x=scores[class_mask, 0],
                y=scores[class_mask, 1],
                z=scores[class_mask, 2],
                mode='markers',
                name=class_name,
                legendgroup=class_name,
                showlegend=(column == 1),  # Avoid duplicating legend entries.
                marker=dict(
                    size=3.5,
                    color=class_colors[class_name],
                    symbol=class_symbols[class_index % len(class_symbols)],
                    opacity=0.65 if split_name == 'Training' else 0.80,
                ),
                # customdata lets hover labels show the row index without using it
                # as a visual encoding.
                customdata=sample_indices[:, np.newaxis],
                hovertemplate=(
                    f'<b>{class_name}</b> — {split_name}<br>'
                    'Sample index: %{customdata[0]}<br>'
                    'PC1: %{x:.2f}<br>'
                    'PC2: %{y:.2f}<br>'
                    'PC3: %{z:.2f}<extra></extra>'
                ),
            ),
            row=1,
            col=column,
        )

# Give both 3D scenes identical axis styling and label each component with the
# percentage of variance it explains. aspectmode='data' preserves PCA distances.
axis_style = dict(
    backgroundcolor='white',
    gridcolor='#D9DEE7',
    zerolinecolor='#AEB7C4',
    showbackground=True,
)
scene_style = dict(
    xaxis=dict(title=f'PC1 ({explained[0]:.1f}%)', **axis_style),
    yaxis=dict(title=f'PC2 ({explained[1]:.1f}%)', **axis_style),
    zaxis=dict(title=f'PC3 ({explained[2]:.1f}%)', **axis_style),
    aspectmode='data',
)

fig.update_layout(
    template='plotly_white',
    height=650,
    title=dict(
        text=(
            '3D PCA of convolutional-autoencoder bottleneck features'
            f'<br><sup>PCA fitted on training features; first three PCs explain '
            f'{explained.sum():.1f}% of training variance</sup>'
        ),
        x=0.5,
    ),
    legend=dict(
        title='Fruit class', orientation='h', y=-0.05, x=0.5, xanchor='center'
    ),
    margin=dict(l=0, r=0, b=65, t=110),
    scene=scene_style,
    scene2=scene_style,
)

# Plotly enables rotation, zoom, hover, and legend filtering directly in Jupyter.
fig.show(config={'displaylogo': False, 'scrollZoom': True})

print('Bottleneck matrix:', ttfruit_train_features.shape)
print(f'Variance explained by PC1 + PC2 + PC3: {explained.sum():.1f}%')

#### How to use and interpret the interactive PCA plot

- **Rotate** either panel by dragging and **zoom** with the mouse wheel.
- **Hover** over a point to see its fruit, split, sample index, and three PCA coordinates.
- **Click** a legend entry to hide/show a fruit; double-click to isolate one class.
- Compare training and test panels. Similar cluster locations suggest that the learned representation transfers consistently to unseen spectra.
- Separated fruit clusters indicate that reconstruction features also contain information related to fruit identity. Overlap does not mean the autoencoder failed—it was optimized for reconstruction, not classification.
- Read the percentages on the axes and in the subtitle. Three components display only part of the bottleneck variance, and a 3D viewing angle can visually exaggerate or hide overlap.

This visualization is exploratory. If class separability must be quantified, complement it with a metric or a classifier evaluated on held-out bottleneck features.